# Entrenamiento final sagital SPIDER desde GCS

Este notebook ejecuta el entrenamiento final reanudable del modelo sagital SPIDER. No entrena el modelo axial. El dataset se lee desde Google Cloud Storage y GCS permanece estrictamente read-only: solamente se consultan metadatos y se descargan objetos. Las imágenes y máscaras se cachean en almacenamiento local antes de entrenar.

Train, validation y test se separan por paciente. Validation se usa para selección de modelo, scheduler y early stopping. Test se evalúa una sola vez con el mejor checkpoint. Las métricas resultantes no constituyen validación clínica; el modelo sigue siendo una herramienta académica de apoyo y requiere validación profesional externa. El entrenamiento requiere GPU y puede reanudarse desde el último epoch completo.

## Seguridad operativa

El notebook está bloqueado por defecto. Valida preflight y smoke test, calcula un token determinístico y se detiene antes de descargar los 894 objetos si `RUN_FINAL_TRAINING` no está activo o si `CONFIRM_FINAL_TRAINING` no coincide. No guarda checkpoints ni resultados parciales hasta superar ese gate.

In [ ]:
import os

os.environ["PFI_REPO_REF"] = (
    "93af43519a8333324f235d194937bdfdd7db9c24"
)

os.environ["PFI_BASE_CHANNELS"] = "16"
os.environ["PFI_BATCH_SIZE"] = "8"
os.environ["PFI_MAX_EPOCHS"] = "80"
os.environ["PFI_NUM_WORKERS"] = "2"
os.environ["PFI_USE_AMP"] = "1"
os.environ["PFI_RESUME_TRAINING"] = "1"

os.environ["RUN_FINAL_TRAINING"] = "1"
os.environ["CONFIRM_FINAL_TRAINING"] = (
    "START-SPIDER-FINAL-fa54c89a-2720b721-b16-e80"
)

print({
    "RUN_FINAL_TRAINING": os.environ["RUN_FINAL_TRAINING"],
    "PFI_REPO_REF": os.environ["PFI_REPO_REF"],
})

In [ ]:
from __future__ import annotations

import importlib.util
import subprocess
import sys

REQUIRED_MODULES = {
    "google.cloud.storage": "google-cloud-storage",
    "google_crc32c": "google-crc32c",
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "pillow",
    "SimpleITK": "SimpleITK",
    "sklearn": "scikit-learn",
    "torch": "torch",
    "matplotlib": "matplotlib",
}

missing = []
for module_name, package_name in REQUIRED_MODULES.items():
    if importlib.util.find_spec(module_name) is None:
        missing.append(package_name)
if missing:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        *sorted(set(missing)),
    ])

import base64
import csv
import hashlib
import json
import math
import multiprocessing
import os
import random
import shutil
import tempfile
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath
from typing import Any

import google.auth
import google_crc32c
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import SimpleITK as sitk
import torch
import torch.nn.functional as F
from google.cloud import storage
from PIL import Image, ImageEnhance
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset

## Repositorio y arquitectura

Clona o actualiza el AI Module, exige árbol limpio, registra commit y SHA de `model_architectures.py`, e importa exclusivamente `SagittalUNet2D` y `build_checkpoint_model`.

In [ ]:
PFI_REPO_URL = "https://github.com/EnzoAA004/PFI_MVPTest_Enzo_AImodule.git"
PFI_REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")
PFI_REPO_REF = os.getenv("PFI_REPO_REF", "main")

def run_git(args: list[str], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(
        ["git", *args],
        cwd=str(cwd) if cwd else None,
        check=check,
        text=True,
        capture_output=True,
    )

def ref_is_branch(repo_root: Path, ref: str) -> bool:
    result = run_git(["rev-parse", "--verify", f"origin/{ref}"], repo_root, check=False)
    return result.returncode == 0

if not PFI_REPO_ROOT.exists():
    run_git(["clone", PFI_REPO_URL, str(PFI_REPO_ROOT)])
    run_git(["checkout", PFI_REPO_REF], PFI_REPO_ROOT)
else:
    status_before = run_git(["status", "--porcelain"], PFI_REPO_ROOT).stdout.strip()
    if status_before:
        raise RuntimeError("El repositorio AI local tiene cambios sin commit; resolver antes de entrenar.")
    run_git(["fetch", "origin"], PFI_REPO_ROOT)
    run_git(["checkout", PFI_REPO_REF], PFI_REPO_ROOT)
    if ref_is_branch(PFI_REPO_ROOT, PFI_REPO_REF):
        run_git(["pull", "--ff-only", "origin", PFI_REPO_REF], PFI_REPO_ROOT)

ARCHITECTURES_PATH = PFI_REPO_ROOT / "ai_service" / "pfi_ai_service" / "model_architectures.py"
if not ARCHITECTURES_PATH.exists():
    raise FileNotFoundError("No existe ai_service/pfi_ai_service/model_architectures.py")

sys.path.insert(0, str(PFI_REPO_ROOT / "ai_service"))
from pfi_ai_service.model_architectures import (
    SagittalUNet2D,
    build_checkpoint_model,
)

def sha256_stream(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            getattr(digest, "update")(chunk)
    return digest.hexdigest()

REPOSITORY_COMMIT = run_git(["rev-parse", "HEAD"], PFI_REPO_ROOT).stdout.strip()
REPOSITORY_STATUS = run_git(["status", "--porcelain"], PFI_REPO_ROOT).stdout.strip()
MODEL_ARCHITECTURE_SHA256 = sha256_stream(ARCHITECTURES_PATH)
if REPOSITORY_STATUS:
    raise RuntimeError("El repositorio AI quedo con cambios locales luego del checkout.")
print(json.dumps({
    "repository_commit": REPOSITORY_COMMIT,
    "repository_clean": REPOSITORY_STATUS == "",
    "model_architecture_sha256": MODEL_ARCHITECTURE_SHA256,
}, indent=2))

## Configuración Colab/VM

Detecta Colab, monta Drive de forma robusta o exige rutas por entorno en VM. Los límites críticos y GCS read-only quedan protegidos por assertions.

In [ ]:
def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

IN_COLAB = running_in_colab()

def mount_drive_if_needed() -> None:
    if not IN_COLAB:
        return
    drive_root = Path("/content/drive")
    my_drive = drive_root / "MyDrive"
    if my_drive.exists():
        return
    from google.colab import drive  # type: ignore
    if drive_root.exists() and any(drive_root.iterdir()):
        shutil.rmtree(drive_root)
    drive.mount(str(drive_root))
    if not my_drive.exists():
        raise RuntimeError("Google Drive no quedo montado en /content/drive/MyDrive")

mount_drive_if_needed()

def getenv_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}

def bounded_int(name: str, default: int, low: int, high: int) -> int:
    value = int(os.getenv(name, str(default)))
    if value < low or value > high:
        raise ValueError(f"{name} debe estar entre {low} y {high}; recibido {value}")
    return value

def env_path(name: str, default: Path | None) -> Path:
    value = os.getenv(name)
    if value:
        return Path(value)
    if default is not None:
        return default
    raise RuntimeError(f"Fuera de Colab debe definirse {name}")

def optional_env_path(
    name: str,
    default: Path | None = None,
) -> Path | None:
    value = os.getenv(name)
    if value:
        return Path(value)
    return default

DEFAULT_CONTROL_DIR = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_training_preflight") if IN_COLAB else None
DEFAULT_SMOKE_DIR = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_spider_smoke_test") if IN_COLAB else None
DEFAULT_OUTPUT_DIR = Path("/content/drive/MyDrive/PFI_MVP/results/GCS_spider_final_training_v1") if IN_COLAB else None
DEFAULT_LOCAL_CACHE_DIR = Path("/content/pfi_gcs_spider_final_cache") if IN_COLAB else None

@dataclass(frozen=True)
class FinalTrainConfig:
    PROJECT_ID: str = "pfi-asplanatti-fabrello-v1"
    BUCKET_NAME: str = "pfi-rm-lumbar-artifacts-2026-ef"
    EXPECTED_MANIFEST_SHA256: str = "fa54c89a278d850021c0f91c0a27b3b5211c86301c9e4f125d75d517f39b793b"
    EXPECTED_TRAINING_INDEX_SHA256: str = "2720b7218c92870f6f0a000b57111ed36b5cf3b78c716f244f427ca7fee4a4ba"
    EXPECTED_INDEX_ROWS: int = 447
    EXPECTED_SPIDER_OBJECTS: int = 894
    EXPECTED_UNIQUE_PATIENTS: int = 218
    TARGET_SIZE: tuple[int, int] = (256, 256)
    NUM_CLASSES: int = 4
    BASE_CHANNELS: int = bounded_int("PFI_BASE_CHANNELS", 16, 8, 32)
    BATCH_SIZE: int = bounded_int("PFI_BATCH_SIZE", 8, 1, 16)
    MAX_EPOCHS: int = bounded_int("PFI_MAX_EPOCHS", 80, 1, 120)
    EARLY_STOPPING_PATIENCE: int = 12
    LEARNING_RATE: float = 0.0008
    WEIGHT_DECAY: float = 0.0001
    SCHEDULER_PATIENCE: int = 4
    SCHEDULER_FACTOR: float = 0.5
    GRADIENT_CLIP_NORM: float = 1.0
    SEED: int = 20260716
    NUM_WORKERS: int = bounded_int("PFI_NUM_WORKERS", 2, 0, 8)
    MAX_FOREGROUND_SLICES_PER_VOLUME: int = 16
    BACKGROUND_RATIO: float = 0.20
    MAX_BACKGROUND_IF_NO_FOREGROUND: int = 4
    TRAIN_LOG_EVERY_N_BATCHES: int = 25
    USE_AMP: bool = getenv_bool("PFI_USE_AMP", True)
    REQUIRE_GPU: bool = True
    RESUME_TRAINING: bool = getenv_bool("PFI_RESUME_TRAINING", True)
    ALLOW_GCS_WRITE: bool = False
    RUN_FINAL_TRAINING: bool = getenv_bool("RUN_FINAL_TRAINING", False)
    CONFIRM_FINAL_TRAINING: str = os.getenv("CONFIRM_FINAL_TRAINING", "")
    CONTROL_DIR: Path = env_path("PFI_CONTROL_DIR", DEFAULT_CONTROL_DIR)
    SMOKE_DIR: Path = env_path("PFI_SMOKE_DIR", DEFAULT_SMOKE_DIR)
    OUTPUT_DIR: Path = env_path("PFI_TRAIN_OUTPUT_DIR", DEFAULT_OUTPUT_DIR)
    LOCAL_CACHE_DIR: Path = env_path("PFI_LOCAL_CACHE_DIR", DEFAULT_LOCAL_CACHE_DIR)

CFG = FinalTrainConfig()
EXPECTED_PATIENT_COUNTS = {"train": 152, "val": 33, "test": 33}
LEGACY_V4_SLICE_COUNTS = {"train": 5535, "val": 1245, "test": 1277}
CLASS_NAMES = ["background", "vertebra_group", "canal", "disc_group"]
SPIDER_SAGITTAL_AXIS = 2
QUALITY_GATE_TARGET_DICE_MACRO_NO_BG = 0.70
GCS_WRITE_OPERATIONS = 0

assert CFG.ALLOW_GCS_WRITE is False
assert CFG.REQUIRE_GPU is True
assert CFG.NUM_CLASSES == 4
assert CFG.TARGET_SIZE == (256, 256)
assert CFG.BASE_CHANNELS in {8, 16, 32}
assert 1 <= CFG.BATCH_SIZE <= 16
assert 1 <= CFG.MAX_EPOCHS <= 120
assert 0 <= CFG.NUM_WORKERS <= 8

EXPECTED_CONFIRMATION_TOKEN = (
    f"START-SPIDER-FINAL-{CFG.EXPECTED_MANIFEST_SHA256[:8]}-"
    f"{CFG.EXPECTED_TRAINING_INDEX_SHA256[:8]}-b{CFG.BASE_CHANNELS}-e{CFG.MAX_EPOCHS}"
)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(CFG.SEED)

def atomic_write_text(path: Path, text: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=path.name, suffix=".tmp", dir=str(path.parent))
    with os.fdopen(fd, "w", encoding="utf-8") as fh:
        fh.write(text)
    os.replace(tmp_name, path)

def atomic_write_json(path: Path, payload: dict[str, Any]) -> None:
    atomic_write_text(path, json.dumps(payload, indent=2, ensure_ascii=False) + "\n")

def atomic_write_csv(path: Path, frame: pd.DataFrame) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    os.replace(tmp, path)

def training_config_payload(
    config: FinalTrainConfig,
) -> dict[str, Any]:
    return {
        "TARGET_SIZE": list(config.TARGET_SIZE),
        "NUM_CLASSES": config.NUM_CLASSES,
        "BASE_CHANNELS": config.BASE_CHANNELS,
        "BATCH_SIZE": config.BATCH_SIZE,
        "MAX_EPOCHS": config.MAX_EPOCHS,
        "EARLY_STOPPING_PATIENCE": config.EARLY_STOPPING_PATIENCE,
        "LEARNING_RATE": config.LEARNING_RATE,
        "WEIGHT_DECAY": config.WEIGHT_DECAY,
        "SCHEDULER_PATIENCE": config.SCHEDULER_PATIENCE,
        "SCHEDULER_FACTOR": config.SCHEDULER_FACTOR,
        "GRADIENT_CLIP_NORM": config.GRADIENT_CLIP_NORM,
        "SEED": config.SEED,
        "NUM_WORKERS": config.NUM_WORKERS,
        "MAX_FOREGROUND_SLICES_PER_VOLUME": config.MAX_FOREGROUND_SLICES_PER_VOLUME,
        "BACKGROUND_RATIO": config.BACKGROUND_RATIO,
        "MAX_BACKGROUND_IF_NO_FOREGROUND": config.MAX_BACKGROUND_IF_NO_FOREGROUND,
        "USE_AMP": config.USE_AMP,
        "loss_algorithm": "0.5*weighted_cross_entropy + 0.5*dice_loss_no_background",
        "augmentation_strategy": "deterministic_epoch_seeded_flips_rotation_contrast_train_only",
        "slice_strategy": "sagittal_axis_2_linspace_max16_foreground_background_ratio_0.20_max4_if_no_foreground",
        "dataset_backend": "preprocessed_uint8_memmap_v1",
    }

def config_fingerprint(training_config: dict[str, Any]) -> str:
    encoded = json.dumps(training_config, sort_keys=True, default=str).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

TRAINING_CONFIG = training_config_payload(CFG)
CONFIG_FINGERPRINT = config_fingerprint(TRAINING_CONFIG)
print(json.dumps({
    "run_final_training": CFG.RUN_FINAL_TRAINING,
    "resume_training": CFG.RESUME_TRAINING,
    "base_channels": CFG.BASE_CHANNELS,
    "max_epochs": CFG.MAX_EPOCHS,
    "batch_size": CFG.BATCH_SIZE,
    "expected_confirmation_token": EXPECTED_CONFIRMATION_TOKEN,
}, indent=2))

## Controles previos y confirmación explícita

Valida los artefactos de preflight y smoke test. Si el token no coincide, el flujo se detiene antes del cache completo y del entrenamiento.

In [ ]:
def read_json(path: Path) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f"Falta archivo de control: {path.name}")
    return json.loads(path.read_text(encoding="utf-8"))

INDEX_CSV = CFG.CONTROL_DIR / "spider_training_index.csv"
PREFLIGHT_SUMMARY_JSON = CFG.CONTROL_DIR / "gcs_training_preflight_summary.json"
PREFLIGHT_EVIDENCE_JSON = CFG.CONTROL_DIR / "gcs_training_preflight_evidence.json"
SMOKE_SUMMARY_JSON = CFG.SMOKE_DIR / "gcs_spider_smoke_test_summary.json"
SMOKE_EVIDENCE_JSON = CFG.SMOKE_DIR / "gcs_spider_smoke_test_evidence.json"

preflight_summary = read_json(PREFLIGHT_SUMMARY_JSON)
preflight_evidence = read_json(PREFLIGHT_EVIDENCE_JSON)
smoke_summary = read_json(SMOKE_SUMMARY_JSON)
smoke_evidence = read_json(SMOKE_EVIDENCE_JSON)

training_index_sha256 = sha256_stream(INDEX_CSV)
if preflight_summary.get("TRAINING_PREFLIGHT_SUCCESS") is not True:
    raise RuntimeError("Preflight 43 no esta confirmado como exitoso.")
if smoke_summary.get("SMOKE_TEST_SUCCESS") is not True:
    raise RuntimeError("Smoke test 44 no esta confirmado como exitoso.")
checks = {
    "manifest_sha_preflight": preflight_summary.get("manifest_sha256") == CFG.EXPECTED_MANIFEST_SHA256,
    "manifest_sha_smoke": smoke_summary.get("manifest_sha256") == CFG.EXPECTED_MANIFEST_SHA256,
    "index_sha_expected": training_index_sha256 == CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "index_sha_preflight": preflight_evidence.get("spider_training_index_sha256") == CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "index_sha_smoke": smoke_summary.get("training_index_sha256") == CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "index_rows": preflight_summary.get("spider_complete_pairs") == CFG.EXPECTED_INDEX_ROWS,
    "objects_verified": preflight_summary.get("expected_objects_verified") == 2058,
    "unexpected_objects": preflight_summary.get("unexpected_objects") == 0,
    "missing_objects": preflight_summary.get("missing_objects") == 0,
    "sampled_opened": preflight_summary.get("sampled_spider_pairs_opened") == 3,
    "smoke_downloads": smoke_summary.get("downloaded_objects") == 12,
    "smoke_verified": smoke_summary.get("verified_downloads") == 12,
    "smoke_read_only": smoke_summary.get("gcs_write_operations") == 0,
    "smoke_evidence_read_only": smoke_evidence.get("gcs_read_only") is True,
    "smoke_evidence_verified": smoke_evidence.get("verified_downloads") == 12,
    "smoke_evidence_index_sha": smoke_evidence.get("training_index_sha256") == CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "smoke_evidence_manifest_sha": (
        "manifest_sha256" not in smoke_evidence
        or smoke_evidence.get("manifest_sha256") == CFG.EXPECTED_MANIFEST_SHA256
    ),
}
failed = [name for name, ok in checks.items() if not ok]
if failed:
    raise RuntimeError(f"Controles previos incompatibles: {failed}")

completed_summary_path = CFG.OUTPUT_DIR / "gcs_spider_final_training_summary.json"
allow_retrain_completed = os.getenv("PFI_ALLOW_RETRAIN_COMPLETED", "0") == "1"
if completed_summary_path.exists():
    completed_summary = json.loads(completed_summary_path.read_text(encoding="utf-8"))
    if completed_summary.get("FINAL_TRAINING_SUCCESS") is True and not allow_retrain_completed:
        raise RuntimeError("El entrenamiento final ya esta completo. Definir PFI_ALLOW_RETRAIN_COMPLETED=1 para reentrenar.")

if not CFG.RUN_FINAL_TRAINING or CFG.CONFIRM_FINAL_TRAINING != EXPECTED_CONFIRMATION_TOKEN:
    print(json.dumps({
        "expected_confirmation_token": EXPECTED_CONFIRMATION_TOKEN,
        "run_final_training": CFG.RUN_FINAL_TRAINING,
        "message": "Validaciones previas OK. No se descargara el dataset completo ni se entrenara sin confirmacion explicita.",
    }, indent=2))
    raise RuntimeError("Confirmacion final requerida antes de cachear y entrenar.")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

if CFG.REQUIRE_GPU and DEVICE.type != "cuda":
    raise RuntimeError(
        "Entrenamiento final confirmado, pero CUDA no esta disponible. "
        "Seleccionar un entorno con GPU antes de descargar el dataset completo."
    )

gpu_info = {
    "torch_version": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_version": torch.version.cuda,
    "gpu_count": torch.cuda.device_count(),
    "device": str(DEVICE),
    "gpu_name": "",
    "gpu_memory_total_mb": 0,
    "compute_capability": "",
}
if DEVICE.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    gpu_info["gpu_name"] = torch.cuda.get_device_name(0)
    gpu_info["gpu_memory_total_mb"] = int(props.total_memory / (1024 * 1024))
    gpu_info["compute_capability"] = f"{props.major}.{props.minor}"
print(json.dumps(gpu_info, indent=2))

CFG.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(CFG.OUTPUT_DIR / "resume").mkdir(parents=True, exist_ok=True)
LAST_CHECKPOINT = (
    CFG.OUTPUT_DIR
    / "resume"
    / "sagittal_spider_final_v1.last_checkpoint.pt"
)
BEST_CHECKPOINT = (
    CFG.OUTPUT_DIR
    / "resume"
    / "sagittal_spider_final_v1.best_checkpoint.pt"
)
HISTORY_CSV = CFG.OUTPUT_DIR / "training_history.csv"
HISTORY_JSON = CFG.OUTPUT_DIR / "training_history.json"

last_exists = LAST_CHECKPOINT.exists()
best_exists = BEST_CHECKPOINT.exists()
if last_exists != best_exists:
    orphan = (
        "best sin last"
        if best_exists
        else "last sin best"
    )
    raise RuntimeError(
        f"Estado de checkpoints incompleto: {orphan}. "
        "No se sobrescriben checkpoints huerfanos."
    )
if last_exists and best_exists and not CFG.RESUME_TRAINING:
    raise RuntimeError(
        "Existen checkpoints last y best, pero "
        "RESUME_TRAINING=False. No se sobrescriben."
    )
orphan_checkpoint_guard = True

def cache_relative_path(
    path_value: str | Path,
) -> str:
    normalized = str(path_value).replace("\\", "/")
    parts = PurePosixPath(normalized).parts
    object_positions = [
        index
        for index, part in enumerate(parts)
        if part == "objects"
    ]
    if not object_positions:
        raise RuntimeError(
            "La ruta de cache no contiene el segmento objects."
        )
    start = object_positions[-1]
    relative = PurePosixPath(*parts[start:]).as_posix()
    relative_parts = PurePosixPath(relative).parts
    if len(relative_parts) < 3:
        raise RuntimeError(
            "Ruta relativa de cache incompleta."
        )
    if relative_parts[0] != "objects":
        raise RuntimeError(
            "Ruta de cache fuera de objects."
        )
    if not relative_parts[1].startswith("case_"):
        raise RuntimeError(
            "Directorio de caso de cache invalido."
        )
    return relative

portable_examples = [
    "/content/cache_a/objects/case_000001/image.mha",
    "/mnt/persistent/cache_b/objects/case_000001/image.mha",
    r"C:\cache_c\objects\case_000001\image.mha",
]
portable_results = {
    cache_relative_path(value)
    for value in portable_examples
}
if portable_results != {
    "objects/case_000001/image.mha"
}:
    raise RuntimeError(
        "La normalizacion portable del cache fallo."
    )
portable_cache_path_fingerprints = True

(CFG.LOCAL_CACHE_DIR / "objects").mkdir(parents=True, exist_ok=True)
(CFG.LOCAL_CACHE_DIR / "tmp").mkdir(parents=True, exist_ok=True)
print("Controles previos confirmados. Gate final habilitado.")


## Índice SPIDER y autenticación GCS

Valida las 447 filas del índice y crea un cliente de Storage solo para lectura. No imprime URI completas ni credenciales.

In [ ]:
INDEX_COLUMNS = [
    "pair_id",
    "image_gcs_uri",
    "mask_gcs_uri",
    "image_size_bytes",
    "mask_size_bytes",
    "image_crc32c",
    "mask_crc32c",
]
spider_index_df = pd.read_csv(INDEX_CSV)
missing_columns = [col for col in INDEX_COLUMNS if col not in spider_index_df.columns]
if missing_columns:
    raise KeyError(f"Indice SPIDER incompleto: {missing_columns}")
if len(spider_index_df) != CFG.EXPECTED_INDEX_ROWS:
    raise ValueError("El indice SPIDER no tiene 447 filas.")
if spider_index_df["pair_id"].isna().any() or (spider_index_df["pair_id"].astype(str).str.len() == 0).any():
    raise ValueError("Hay pair_id vacios.")
if spider_index_df["pair_id"].duplicated().any():
    raise ValueError("Hay pair_id duplicados.")
if spider_index_df["image_gcs_uri"].duplicated().any() or spider_index_df["mask_gcs_uri"].duplicated().any():
    raise ValueError("Hay URI duplicadas por tipo.")
prefix = f"gs://{CFG.BUCKET_NAME}/"
for col in ["image_gcs_uri", "mask_gcs_uri"]:
    if not spider_index_df[col].astype(str).str.startswith(prefix).all():
        raise ValueError(f"{col} contiene URI fuera del bucket esperado.")
if (spider_index_df["image_size_bytes"] <= 0).any() or (spider_index_df["mask_size_bytes"] <= 0).any():
    raise ValueError("El indice contiene tamanos no positivos.")
if (spider_index_df["image_crc32c"].astype(str).str.len() == 0).any() or (spider_index_df["mask_crc32c"].astype(str).str.len() == 0).any():
    raise ValueError("El indice contiene CRC32C vacios.")
if (spider_index_df["image_gcs_uri"] == spider_index_df["mask_gcs_uri"]).any():
    raise ValueError("Hay URI de imagen iguales a URI de mascara.")
all_uris = pd.concat([spider_index_df["image_gcs_uri"], spider_index_df["mask_gcs_uri"]], ignore_index=True)
if all_uris.nunique() != CFG.EXPECTED_SPIDER_OBJECTS:
    raise ValueError("No hay exactamente 894 URI SPIDER distintas.")

if IN_COLAB:
    from google.colab import auth  # type: ignore
    auth.authenticate_user()
    credentials, _ = google.auth.default()
else:
    credentials, _ = google.auth.default()
client = storage.Client(project=CFG.PROJECT_ID, credentials=credentials)
bucket = client.bucket(CFG.BUCKET_NAME)
bucket.reload()
if bucket.name != CFG.BUCKET_NAME:
    raise RuntimeError("Bucket inesperado.")
print(json.dumps({
    "index_rows": int(len(spider_index_df)),
    "spider_objects": int(all_uris.nunique()),
    "bucket_name": bucket.name,
    "bucket_location": getattr(bucket, "location", None),
    "bucket_storage_class": getattr(bucket, "storage_class", None),
    "gcs_write_operations": GCS_WRITE_OPERATIONS,
}, indent=2))

## Cache local completa

Descarga o reutiliza los 447 pares SPIDER en directorios anónimos, verificando tamaño y CRC32C con movimiento atómico desde `.partial`.

In [ ]:
def full_suffix(path_or_uri: str) -> str:
    name = Path(path_or_uri.split("/")[-1]).name
    lower = name.lower()
    if lower.endswith(".nii.gz"):
        return ".nii.gz"
    return Path(name).suffix.lower()

def blob_name_from_uri(uri: str) -> str:
    prefix = f"gs://{CFG.BUCKET_NAME}/"
    if not uri.startswith(prefix):
        raise ValueError("URI fuera del bucket esperado.")
    return uri[len(prefix):]

def crc32c_stream_base64(path: Path) -> str:
    checksum = google_crc32c.Checksum()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            getattr(checksum, "update")(chunk)
    return base64.b64encode(checksum.digest()).decode("ascii")

def verify_local_file(path: Path, expected_size: int, expected_crc32c: str) -> bool:
    if not path.exists() or not path.is_file():
        return False
    if path.stat().st_size != int(expected_size):
        return False
    return crc32c_stream_base64(path) == str(expected_crc32c)

def download_verified(uri: str, expected_size: int, expected_crc32c: str, destination: Path) -> str:
    if verify_local_file(destination, expected_size, expected_crc32c):
        return "existing_verified"
    blob = bucket.blob(blob_name_from_uri(uri))
    blob.reload()
    if int(blob.size) != int(expected_size):
        raise RuntimeError("Metadata remota con tamano inesperado.")
    if str(blob.crc32c) != str(expected_crc32c):
        raise RuntimeError("Metadata remota con CRC32C inesperado.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = CFG.LOCAL_CACHE_DIR / "tmp" / f"{destination.name}.partial"
    if tmp_path.exists():
        tmp_path.unlink()
    blob.download_to_filename(str(tmp_path))
    if not verify_local_file(tmp_path, expected_size, expected_crc32c):
        raise RuntimeError("Descarga corrupta o incompleta.")
    os.replace(tmp_path, destination)
    if not verify_local_file(destination, expected_size, expected_crc32c):
        raise RuntimeError("Archivo final corrupto tras movimiento atomico.")
    return "downloaded_verified"

receipt_rows = []
for pair_index, row in spider_index_df.reset_index(drop=True).iterrows():
    case_dir = CFG.LOCAL_CACHE_DIR / "objects" / f"case_{pair_index:06d}"
    image_path = case_dir / f"image{full_suffix(row.image_gcs_uri)}"
    mask_path = case_dir / f"mask{full_suffix(row.mask_gcs_uri)}"
    image_status = download_verified(row.image_gcs_uri, int(row.image_size_bytes), row.image_crc32c, image_path)
    mask_status = download_verified(row.mask_gcs_uri, int(row.mask_size_bytes), row.mask_crc32c, mask_path)
    receipt_rows.append({
        "pair_index": int(pair_index),
        "image_local_path": str(image_path),
        "mask_local_path": str(mask_path),
        "image_size_bytes": int(row.image_size_bytes),
        "mask_size_bytes": int(row.mask_size_bytes),
        "image_crc32c": str(row.image_crc32c),
        "mask_crc32c": str(row.mask_crc32c),
        "image_cache_status": image_status,
        "mask_cache_status": mask_status,
    })
    done = pair_index + 1
    if done % 25 == 0 or done == CFG.EXPECTED_INDEX_ROWS:
        print(f"Cache SPIDER verificado: {done}/{CFG.EXPECTED_INDEX_ROWS}")

cache_receipt_df = pd.DataFrame(receipt_rows)
allowed_status = {"downloaded_verified", "existing_verified"}
if len(cache_receipt_df) != CFG.EXPECTED_INDEX_ROWS:
    raise RuntimeError("Receipt de cache incompleto.")
if not set(cache_receipt_df["image_cache_status"]).issubset(allowed_status):
    raise RuntimeError("Estado de cache de imagen invalido.")
if not set(cache_receipt_df["mask_cache_status"]).issubset(allowed_status):
    raise RuntimeError("Estado de cache de mascara invalido.")
verified_cache_objects = int(len(cache_receipt_df) * 2)
if verified_cache_objects != CFG.EXPECTED_SPIDER_OBJECTS:
    raise RuntimeError("No se verificaron los 894 objetos.")
def canonical_records_sha256(records: list[dict[str, Any]]) -> str:
    canonical_json = json.dumps(
        records,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")
    return hashlib.sha256(canonical_json).hexdigest()

CACHE_RECEIPT_CSV = CFG.OUTPUT_DIR / "spider_cache_receipt.csv"
atomic_write_csv(CACHE_RECEIPT_CSV, cache_receipt_df)
CACHE_RECEIPT_FILE_SHA256 = sha256_stream(CACHE_RECEIPT_CSV)
cache_content_records = []
for row in cache_receipt_df.sort_values("pair_index").itertuples(index=False):
    cache_content_records.append({
        "pair_index": int(row.pair_index),
        "image_relative_path": cache_relative_path(row.image_local_path),
        "mask_relative_path": cache_relative_path(row.mask_local_path),
        "image_size_bytes": int(row.image_size_bytes),
        "mask_size_bytes": int(row.mask_size_bytes),
        "image_crc32c": str(row.image_crc32c),
        "mask_crc32c": str(row.mask_crc32c),
    })
CACHE_CONTENT_SHA256 = canonical_records_sha256(cache_content_records)
print(json.dumps({
    "cached_pairs": len(cache_receipt_df),
    "verified_cache_objects": verified_cache_objects,
    "cache_content_sha256": CACHE_CONTENT_SHA256,
}, indent=2))


## Formatos médicos y mapping de etiquetas

Lee `.npy`, imágenes comunes, `.mha/.mhd` con SimpleITK y NIfTI bajo demanda. Aplica el mapping SPIDER a cuatro clases sin inventar traducciones.

In [ ]:
def ensure_nibabel_if_needed(path: Path):
    lower = path.name.lower()
    if lower.endswith(".nii") or lower.endswith(".nii.gz"):
        if importlib.util.find_spec("nibabel") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "nibabel"])
        import nibabel as nib  # type: ignore
        return nib
    return None

def read_array(path: Path) -> np.ndarray:
    suffix = full_suffix(str(path))
    try:
        if suffix == ".npy":
            return np.asarray(np.load(path))
        if suffix in {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}:
            return np.asarray(Image.open(path).convert("L"))
        if suffix in {".mha", ".mhd"}:
            image = sitk.ReadImage(str(path))
            return np.asarray(sitk.GetArrayFromImage(image))
        if suffix in {".nii", ".nii.gz"}:
            nib = ensure_nibabel_if_needed(path)
            return np.asarray(nib.load(str(path)).get_fdata())
    except RuntimeError as exc:
        if suffix == ".mhd":
            raise RuntimeError("No se pudo abrir un .mhd; verificar que su sidecar este disponible en el cache local.") from exc
        raise
    raise ValueError(f"Formato no soportado: {suffix}")

def canonicalize_spider_array(arr):
    arr = np.asarray(arr)
    if arr.ndim == 3 and arr.shape[0] <= 64 and arr.shape[-1] > 64:
        return np.moveaxis(arr, 0, -1)
    return arr

def _int_value(value: Any) -> int:
    if isinstance(value, (np.integer, int)):
        return int(value)
    if isinstance(value, float) and value.is_integer():
        return int(value)
    if isinstance(value, str):
        stripped = value.strip()
        if stripped.endswith(".0"):
            stripped = stripped[:-2]
        return int(stripped)
    raise ValueError(f"No se puede convertir a entero: {value!r}")

def normalize_label_mapping(raw_mapping: dict[str, Any]) -> dict[int, int]:
    mapping: dict[int, int] = {}
    for key, value in raw_mapping.items():
        key_int = _int_value(key)
        if isinstance(value, list):
            class_id = key_int
            for raw_label in value:
                mapping[_int_value(raw_label)] = class_id
        else:
            mapping[key_int] = _int_value(value)
    output_classes = set(mapping.values())
    if output_classes != {0, 1, 2, 3}:
        raise ValueError(f"Mapping SPIDER debe producir clases 0..3; recibido {sorted(output_classes)}")
    return dict(sorted(mapping.items()))

def load_json_mapping(path: Path) -> dict[int, int]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    if "label_group_mapping" in payload:
        payload = payload["label_group_mapping"]
    return normalize_label_mapping(payload)

def load_spider_label_map() -> dict[int, int]:
    default_json = Path("/content/drive/MyDrive/PFI_MVP/results/E5_multiclase_agrupado/E5_multiclass_label_mapping.json") if IN_COLAB else None
    default_checkpoint = Path("/content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt") if IN_COLAB else None
    mapping_json = optional_env_path("PFI_SPIDER_LABEL_MAPPING_JSON", default_json)
    checkpoint_path = optional_env_path("PFI_SPIDER_MAPPING_CHECKPOINT", default_checkpoint)
    if mapping_json is not None and mapping_json.exists():
        return load_json_mapping(mapping_json)
    if checkpoint_path is not None and checkpoint_path.exists():
        checkpoint = torch.load(
            checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )
        if "label_group_mapping" not in checkpoint:
            raise KeyError("El checkpoint fallback no contiene label_group_mapping.")
        return normalize_label_mapping(checkpoint["label_group_mapping"])
    searched = {
        "PFI_SPIDER_LABEL_MAPPING_JSON": str(mapping_json) if mapping_json is not None else None,
        "PFI_SPIDER_MAPPING_CHECKPOINT": str(checkpoint_path) if checkpoint_path is not None else None,
    }
    raise FileNotFoundError(
        "No se encontro mapping SPIDER. Definir PFI_SPIDER_LABEL_MAPPING_JSON "
        "con un JSON valido; PFI_SPIDER_MAPPING_CHECKPOINT es solo fallback opcional. "
        f"Rutas evaluadas: {searched}"
    )

def apply_label_map(mask: np.ndarray, label_map: dict[int, int], case_label: str = "case") -> np.ndarray:
    labels = sorted(_int_value(v) for v in np.unique(mask))
    missing = [v for v in labels if v not in label_map]
    if missing:
        raise ValueError(f"{case_label}: labels sin mapping {missing[:10]}")
    out = np.zeros(mask.shape, dtype=np.int64)
    for raw_label, class_id in label_map.items():
        out[np.asarray(mask) == raw_label] = class_id
    unique_out = set(int(v) for v in np.unique(out))
    if not unique_out.issubset({0, 1, 2, 3}):
        raise ValueError("Mapping produjo clases fuera de 0..3.")
    return out

SPIDER_LABEL_MAP = load_spider_label_map()
MAPPING_JSON = CFG.OUTPUT_DIR / "spider_label_group_mapping.json"
atomic_write_json(MAPPING_JSON, {str(k): int(v) for k, v in SPIDER_LABEL_MAP.items()})
MAPPING_SHA256 = sha256_stream(MAPPING_JSON)
print(json.dumps({"mapping_classes": sorted(set(SPIDER_LABEL_MAP.values())), "mapping_sha256": MAPPING_SHA256}, indent=2))

## Split por paciente, slices y class weights

Congela splits por paciente, construye registros de slices determinísticos y calcula pesos de clase exactos desde los píxeles de entrenamiento.

In [ ]:
def patient_id_from_stem(stem: str) -> str:
    return stem.replace("-", "_").split("_")[0] or stem

def canonical_patient_split_records(
    frame: pd.DataFrame,
) -> list[dict[str, Any]]:
    normalized = frame[
        ["patient_id", "split"]
    ].copy()

    normalized["patient_id"] = (
        normalized["patient_id"]
        .astype("string")
    )

    normalized["split"] = (
        normalized["split"]
        .astype("string")
    )

    normalized = normalized.sort_values(
        ["split", "patient_id"],
        kind="mergesort",
    ).reset_index(drop=True)

    return [
        {
            "patient_id": str(row.patient_id),
            "split": str(row.split),
        }
        for row in normalized.itertuples(index=False)
    ]

_patient_split_numeric = pd.DataFrame({
    "patient_id": [1, 2, 10, 11],
    "split": ["train", "train", "train", "train"],
})

_patient_split_text = pd.DataFrame({
    "patient_id": ["1", "2", "10", "11"],
    "split": ["train", "train", "train", "train"],
})

if (
    canonical_patient_split_records(
        _patient_split_numeric
    )
    != canonical_patient_split_records(
        _patient_split_text
    )
):
    raise RuntimeError(
        "La canonicalizacion portable de patient_id fallo."
    )

del _patient_split_numeric
del _patient_split_text

def canonical_pair_split_records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    for row in frame.sort_values(["split", "pair_index"]).itertuples(index=False):
        records.append({
            "pair_index": int(row.pair_index),
            "pair_id": str(row.pair_id),
            "patient_id": str(row.patient_id),
            "split": str(row.split),
            "image_relative_path": cache_relative_path(row.image_local_path),
            "mask_relative_path": cache_relative_path(row.mask_local_path),
        })
    return records

def canonical_slice_records(frame: pd.DataFrame) -> list[dict[str, Any]]:
    records = []
    ordered = frame.sort_values(["split", "pair_index", "slice_axis", "slice_index", "has_foreground"])
    for row in ordered.itertuples(index=False):
        records.append({
            "split": str(row.split),
            "pair_index": int(row.pair_index),
            "patient_id": str(row.patient_id),
            "image_relative_path": cache_relative_path(row.image_local_path),
            "mask_relative_path": cache_relative_path(row.mask_local_path),
            "slice_index": int(row.slice_index),
            "slice_axis": int(row.slice_axis),
            "has_foreground": bool(row.has_foreground),
        })
    return records

def compare_frozen_records(
    path: Path,
    expected_frame: pd.DataFrame,
    canonicalizer,
    label: str,
    read_csv_dtype: dict[str, str] | None = None,
) -> bool:
    if not path.exists():
        atomic_write_csv(path, expected_frame)
        return True

    frozen_frame = pd.read_csv(
        path,
        dtype=read_csv_dtype,
    )

    frozen_records = canonicalizer(
        frozen_frame
    )

    expected_records = canonicalizer(
        expected_frame
    )

    if frozen_records != expected_records:
        raise RuntimeError(
            f"{label} congelado difiere de la "
            "reconstruccion deterministica."
        )

    return True

pair_df = cache_receipt_df.copy()
pair_df["pair_id"] = spider_index_df["pair_id"].astype(str).values
pair_df["patient_id"] = pair_df["pair_id"].map(lambda value: patient_id_from_stem(Path(value).stem))
if pair_df["patient_id"].isna().any() or (pair_df["patient_id"].astype(str).str.len() == 0).any():
    raise RuntimeError("Hay pair_id sin patient_id derivable.")
if pair_df.groupby("pair_id")["patient_id"].nunique().max() != 1:
    raise RuntimeError("Un pair_id pertenece a mas de un patient_id.")
patient_ids = sorted(pair_df["patient_id"].unique())
if len(patient_ids) != CFG.EXPECTED_UNIQUE_PATIENTS:
    raise RuntimeError("Cantidad de pacientes SPIDER inesperada.")

train_ids, temp_ids = train_test_split(
    patient_ids,
    test_size=0.30,
    random_state=CFG.SEED,
)
val_ids, test_ids = train_test_split(
    temp_ids,
    test_size=0.50,
    random_state=CFG.SEED,
)
split_rows = [{"patient_id": pid, "split": "train"} for pid in sorted(train_ids)]
split_rows += [{"patient_id": pid, "split": "val"} for pid in sorted(val_ids)]
split_rows += [{"patient_id": pid, "split": "test"} for pid in sorted(test_ids)]
expected_patient_split_df = pd.DataFrame(split_rows).sort_values(["split", "patient_id"]).reset_index(drop=True)

PATIENT_SPLIT_CSV = CFG.OUTPUT_DIR / "spider_patient_split.csv"
patient_split_matches_reconstruction = (
    compare_frozen_records(
        PATIENT_SPLIT_CSV,
        expected_patient_split_df,
        canonical_patient_split_records,
        "spider_patient_split.csv",
        read_csv_dtype={
            "patient_id": "string",
            "split": "string",
        },
    )
)
patient_split_df = expected_patient_split_df

if set(patient_split_df.columns) != {"patient_id", "split"}:
    raise RuntimeError("spider_patient_split.csv tiene columnas inesperadas.")
if patient_split_df["patient_id"].duplicated().any():
    raise RuntimeError("Hay pacientes duplicados en el split congelado.")
if set(patient_split_df["patient_id"]) != set(patient_ids):
    raise RuntimeError("El split congelado no coincide con los pacientes actuales.")
patient_counts = patient_split_df["split"].value_counts().to_dict()
if patient_counts != EXPECTED_PATIENT_COUNTS:
    raise RuntimeError(f"Conteos de pacientes inesperados: {patient_counts}")
split_sets = {split: set(patient_split_df.loc[patient_split_df["split"] == split, "patient_id"]) for split in EXPECTED_PATIENT_COUNTS}
if split_sets["train"] & split_sets["val"] or split_sets["train"] & split_sets["test"] or split_sets["val"] & split_sets["test"]:
    raise RuntimeError("Leakage de pacientes entre splits.")
PATIENT_SPLIT_SHA256 = sha256_stream(PATIENT_SPLIT_CSV)

pair_split_df = pair_df.merge(patient_split_df, on="patient_id", how="left", validate="many_to_one")
if pair_split_df["split"].isna().any():
    raise RuntimeError("Hay pares sin split asignado.")
PAIR_SPLIT_CSV = CFG.OUTPUT_DIR / "spider_pair_split.csv"
pair_split_export = pair_split_df[["pair_index", "pair_id", "patient_id", "split", "image_local_path", "mask_local_path"]].sort_values(["split", "pair_index"]).reset_index(drop=True)
pair_split_matches_reconstruction = compare_frozen_records(
    PAIR_SPLIT_CSV,
    pair_split_export,
    canonical_pair_split_records,
    "spider_pair_split.csv",
)
PAIR_SPLIT_FILE_SHA256 = sha256_stream(PAIR_SPLIT_CSV)
PAIR_SPLIT_CONTENT_SHA256 = canonical_records_sha256(canonical_pair_split_records(pair_split_export))

# Usar rutas locales actuales en memoria aunque el CSV congelado provenga de otra VM.
pair_split_df = pair_split_export.copy()

def choose_indices(values: list[int], limit: int) -> list[int]:
    if not values or limit <= 0:
        return []
    ordered = sorted(int(v) for v in values)
    if len(ordered) <= limit:
        return ordered
    positions = np.linspace(0, len(ordered) - 1, num=limit, dtype=int)
    return [ordered[int(pos)] for pos in positions]

def validate_volume_arrays(image_arr: np.ndarray, mask_arr: np.ndarray) -> None:
    if image_arr.size == 0 or mask_arr.size == 0:
        raise RuntimeError("Volumen SPIDER vacio.")
    if image_arr.ndim not in {2, 3} or mask_arr.ndim not in {2, 3}:
        raise RuntimeError("SPIDER debe ser 2D o 3D.")
    if image_arr.shape != mask_arr.shape:
        raise RuntimeError("Imagen y mascara SPIDER con shapes distintas.")
    if not np.isfinite(image_arr).all():
        raise RuntimeError("Imagen SPIDER contiene valores no finitos.")
    if len(np.unique(mask_arr)) == 0:
        raise RuntimeError("Mascara SPIDER sin valores.")
    if image_arr.ndim == 3 and SPIDER_SAGITTAL_AXIS >= image_arr.ndim:
        raise RuntimeError("Eje sagital invalido para volumen 3D.")

slice_rows = []
train_pixel_counts = np.zeros(CFG.NUM_CLASSES, dtype=np.int64)
background_slices_verified_empty = True
for row in pair_split_df.itertuples(index=False):
    image_arr = canonicalize_spider_array(read_array(Path(row.image_local_path)))
    raw_mask_arr = canonicalize_spider_array(read_array(Path(row.mask_local_path)))
    validate_volume_arrays(image_arr, raw_mask_arr)
    mapped_mask = apply_label_map(raw_mask_arr, SPIDER_LABEL_MAP, case_label="spider")
    if mapped_mask.ndim == 2:
        foreground = [0] if np.any(mapped_mask > 0) else []
        background = [] if foreground else [0]
        selected_fg = foreground
        selected_bg = background[:1]
    else:
        foreground = [idx for idx in range(mapped_mask.shape[SPIDER_SAGITTAL_AXIS]) if np.any(np.take(mapped_mask, idx, axis=SPIDER_SAGITTAL_AXIS) > 0)]
        selected_fg = choose_indices(foreground, CFG.MAX_FOREGROUND_SLICES_PER_VOLUME)
        all_foreground_set = set(foreground)
        background_pool = [idx for idx in range(mapped_mask.shape[SPIDER_SAGITTAL_AXIS]) if idx not in all_foreground_set]
        if selected_fg:
            n_background = min(len(background_pool), max(1, int(len(selected_fg) * CFG.BACKGROUND_RATIO)))
        else:
            n_background = min(len(background_pool), CFG.MAX_BACKGROUND_IF_NO_FOREGROUND)
        selected_bg = choose_indices(background_pool, n_background)
    for idx in selected_bg:
        selected_mask = mapped_mask if mapped_mask.ndim == 2 else np.take(
            mapped_mask,
            idx,
            axis=SPIDER_SAGITTAL_AXIS,
        )
        if np.any(selected_mask > 0):
            background_slices_verified_empty = False
            raise RuntimeError(
                "Una slice seleccionada como background contiene foreground."
            )
    for slice_index in selected_fg:
        slice_rows.append({
            "split": row.split,
            "pair_index": int(row.pair_index),
            "patient_id": row.patient_id,
            "image_local_path": row.image_local_path,
            "mask_local_path": row.mask_local_path,
            "slice_index": int(slice_index),
            "slice_axis": SPIDER_SAGITTAL_AXIS if mapped_mask.ndim == 3 else 0,
            "has_foreground": True,
        })
    for slice_index in selected_bg:
        slice_rows.append({
            "split": row.split,
            "pair_index": int(row.pair_index),
            "patient_id": row.patient_id,
            "image_local_path": row.image_local_path,
            "mask_local_path": row.mask_local_path,
            "slice_index": int(slice_index),
            "slice_axis": SPIDER_SAGITTAL_AXIS if mapped_mask.ndim == 3 else 0,
            "has_foreground": False,
        })
    if row.split == "train":
        for record in [r for r in slice_rows if r["pair_index"] == int(row.pair_index) and r["split"] == "train"]:
            sl = mapped_mask if mapped_mask.ndim == 2 else np.take(mapped_mask, record["slice_index"], axis=record["slice_axis"])
            train_pixel_counts += np.bincount(sl.astype(np.int64).ravel(), minlength=CFG.NUM_CLASSES)[:CFG.NUM_CLASSES]

slice_records_df = pd.DataFrame(slice_rows).sort_values(["split", "pair_index", "slice_axis", "slice_index", "has_foreground"]).reset_index(drop=True)
slice_counts = {split: int(slice_records_df.loc[slice_records_df["split"] == split].shape[0]) for split in EXPECTED_PATIENT_COUNTS}
all_splits_have_slices = set(slice_counts) == set(EXPECTED_PATIENT_COUNTS) and all(slice_counts[split] > 0 for split in EXPECTED_PATIENT_COUNTS)
if not all_splits_have_slices:
    raise RuntimeError(f"Algun split quedo sin slices: {slice_counts}")
all_splits_have_foreground = slice_records_df.groupby("split")["has_foreground"].any().reindex(EXPECTED_PATIENT_COUNTS).fillna(False).all()
if not all_splits_have_foreground:
    raise RuntimeError("Algun split no tiene foreground.")
slice_records_no_leakage = True
for split_name, patients in split_sets.items():
    observed = set(slice_records_df.loc[slice_records_df["split"] == split_name, "patient_id"])
    if not observed.issubset(patients):
        slice_records_no_leakage = False
        raise RuntimeError("Leakage detectado en slice_records.")
SLICE_RECORDS_CSV = CFG.OUTPUT_DIR / "spider_slice_records.csv"
slice_records_match_reconstruction = compare_frozen_records(
    SLICE_RECORDS_CSV,
    slice_records_df,
    canonical_slice_records,
    "spider_slice_records.csv",
)
SLICE_RECORDS_FILE_SHA256 = sha256_stream(SLICE_RECORDS_CSV)
SLICE_RECORDS_CONTENT_SHA256 = canonical_records_sha256(canonical_slice_records(slice_records_df))
slice_records_df["cache_row_index"] = np.arange(
    len(slice_records_df),
    dtype=np.int64,
)
slice_counts_match_legacy_v4 = slice_counts == LEGACY_V4_SLICE_COUNTS

if np.any(train_pixel_counts <= 0):
    raise RuntimeError("Las cuatro clases deben tener pixeles positivos en train.")
raw_weights = train_pixel_counts.sum() / train_pixel_counts.astype(np.float64)
normalized_weights = raw_weights / raw_weights.mean()
clipped_weights = np.clip(normalized_weights, 0.25, 8.0)
if not np.isfinite(clipped_weights).all():
    raise RuntimeError("Class weights no finitos.")
CLASS_DISTRIBUTION_JSON = CFG.OUTPUT_DIR / "spider_class_distribution.json"
atomic_write_json(CLASS_DISTRIBUTION_JSON, {
    "class_names": CLASS_NAMES,
    "pixel_counts": {CLASS_NAMES[i]: int(train_pixel_counts[i]) for i in range(CFG.NUM_CLASSES)},
    "raw_weights": {CLASS_NAMES[i]: float(normalized_weights[i]) for i in range(CFG.NUM_CLASSES)},
    "clipped_weights": {CLASS_NAMES[i]: float(clipped_weights[i]) for i in range(CFG.NUM_CLASSES)},
    "total_pixels": int(train_pixel_counts.sum()),
    "source_slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
    "source_slice_records_file_sha256": SLICE_RECORDS_FILE_SHA256,
})
CLASS_DISTRIBUTION_SHA256 = sha256_stream(CLASS_DISTRIBUTION_JSON)
CLASS_WEIGHTS_TENSOR = torch.tensor(clipped_weights, dtype=torch.float32)
print(json.dumps({
    "patient_counts": patient_counts,
    "actual_slice_counts": slice_counts,
    "slice_counts_match_legacy_v4": slice_counts_match_legacy_v4,
    "pair_split_content_sha256": PAIR_SPLIT_CONTENT_SHA256,
    "slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
    "class_distribution_sha256": CLASS_DISTRIBUTION_SHA256,
}, indent=2))


## Dataset, preprocesamiento y augmentation

Preprocesa a tensores `[1, 256, 256]`, evita warnings de Pillow y aplica augmentation solo en train.

In [ ]:
def robust_normalize(image: np.ndarray) -> np.ndarray:
    arr = np.asarray(image, dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    p1, p99 = np.percentile(arr, [1, 99])
    if p99 <= p1:
        return np.zeros(arr.shape, dtype=np.float32)
    arr = (arr - p1) / (p99 - p1)
    return np.clip(arr, 0.0, 1.0).astype(np.float32)

def resize_image(image: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    pil = Image.fromarray((np.clip(image, 0.0, 1.0) * 255).astype(np.uint8))
    resized = pil.resize((size[1], size[0]), Image.Resampling.BILINEAR)
    return (np.asarray(resized).astype(np.float32) / 255.0).astype(np.float32)

def resize_image_uint8(image: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    pil = Image.fromarray((np.clip(image, 0.0, 1.0) * 255).astype(np.uint8))
    resized = pil.resize((size[1], size[0]), Image.Resampling.BILINEAR)
    return np.asarray(resized, dtype=np.uint8)

def resize_mask(mask: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    pil = Image.fromarray(mask.astype(np.int32))
    resized = pil.resize((size[1], size[0]), Image.Resampling.NEAREST)
    out = np.asarray(resized, dtype=np.int64)
    if not set(int(v) for v in np.unique(out)).issubset({0, 1, 2, 3}):
        raise RuntimeError("Resize nearest produjo labels fuera de 0..3.")
    return out

def resize_mask_uint8(mask: np.ndarray, size: tuple[int, int]) -> np.ndarray:
    return resize_mask(mask, size).astype(np.uint8)

def extract_slice(arr: np.ndarray, slice_index: int, slice_axis: int) -> np.ndarray:
    arr = canonicalize_spider_array(arr)
    if arr.ndim == 2:
        return arr
    return np.take(arr, int(slice_index), axis=int(slice_axis))

def augment_pair(image: np.ndarray, mask: np.ndarray, rng: random.Random) -> tuple[np.ndarray, np.ndarray]:
    img = Image.fromarray((np.clip(image, 0.0, 1.0) * 255).astype(np.uint8))
    msk = Image.fromarray(mask.astype(np.int32))
    if rng.random() < 0.5:
        img = img.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        msk = msk.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
    if rng.random() < 0.2:
        img = img.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
        msk = msk.transpose(Image.Transpose.FLIP_TOP_BOTTOM)
    if rng.random() < 0.4:
        angle = rng.uniform(-8.0, 8.0)
        img = img.rotate(angle, resample=Image.Resampling.BILINEAR)
        msk = msk.rotate(angle, resample=Image.Resampling.NEAREST)
    if rng.random() < 0.5:
        img = ImageEnhance.Contrast(img).enhance(rng.uniform(0.85, 1.15))
    return np.asarray(img).astype(np.float32) / 255.0, np.asarray(msk, dtype=np.int64)

PREPROCESSED_CACHE_VERSION = "preprocessed_uint8_v1"
DATASET_BACKEND = "preprocessed_uint8_memmap_v1"
PREPROCESSING_ALGORITHM = "robust_percentile_1_99_uint8_bilinear_image_nearest_uint8_mask"
PREPROCESSED_CACHE_DIR = CFG.LOCAL_CACHE_DIR / PREPROCESSED_CACHE_VERSION
PREPROCESSED_IMAGE_PATH = PREPROCESSED_CACHE_DIR / "spider_images_uint8.npy"
PREPROCESSED_MASK_PATH = PREPROCESSED_CACHE_DIR / "spider_masks_uint8.npy"
PREPROCESSED_METADATA_PATH = PREPROCESSED_CACHE_DIR / "spider_preprocessed_cache_metadata.json"

def preprocessed_cache_content_payload(metadata: dict[str, Any]) -> dict[str, Any]:
    return {
        "cache_version": metadata["cache_version"],
        "slice_records_content_sha256": metadata["slice_records_content_sha256"],
        "mapping_sha256": metadata["mapping_sha256"],
        "target_size": metadata["target_size"],
        "image_shape": metadata["image_shape"],
        "mask_shape": metadata["mask_shape"],
        "image_dtype": metadata["image_dtype"],
        "mask_dtype": metadata["mask_dtype"],
        "preprocessing_algorithm": metadata["preprocessing_algorithm"],
        "image_file_sha256": metadata["image_file_sha256"],
        "mask_file_sha256": metadata["mask_file_sha256"],
    }

def preprocessed_cache_content_sha256(metadata: dict[str, Any]) -> str:
    encoded = json.dumps(
        preprocessed_cache_content_payload(metadata),
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    ).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()

def validate_memmap_file(path: Path, expected_shape: tuple[int, int, int], expected_dtype: np.dtype) -> np.memmap:
    if not path.exists() or path.stat().st_size <= 0:
        raise RuntimeError(f"Memmap inexistente o vacio: {path.name}")
    arr = np.load(path, mmap_mode="r")
    if tuple(arr.shape) != expected_shape:
        raise RuntimeError(f"Shape inesperado en {path.name}: {arr.shape}")
    if arr.dtype != expected_dtype:
        raise RuntimeError(f"Dtype inesperado en {path.name}: {arr.dtype}")
    return arr

def validate_mask_values(mask_memmap: np.memmap, block_size: int = 512) -> None:
    for start in range(0, int(mask_memmap.shape[0]), block_size):
        values = set(int(v) for v in np.unique(mask_memmap[start:start + block_size]))
        if not values.issubset({0, 1, 2, 3}):
            raise RuntimeError("El cache preprocesado de mascaras contiene valores fuera de 0..3.")

def expected_preprocessed_static_metadata(total_slices: int) -> dict[str, Any]:
    shape = [int(total_slices), int(CFG.TARGET_SIZE[0]), int(CFG.TARGET_SIZE[1])]
    return {
        "cache_version": PREPROCESSED_CACHE_VERSION,
        "slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
        "mapping_sha256": MAPPING_SHA256,
        "target_size": list(CFG.TARGET_SIZE),
        "image_shape": shape,
        "mask_shape": shape,
        "image_dtype": "uint8",
        "mask_dtype": "uint8",
        "total_slices": int(total_slices),
        "preprocessing_algorithm": PREPROCESSING_ALGORITHM,
    }

def validate_existing_preprocessed_cache() -> tuple[bool, dict[str, Any] | None]:
    if not (PREPROCESSED_IMAGE_PATH.exists() and PREPROCESSED_MASK_PATH.exists() and PREPROCESSED_METADATA_PATH.exists()):
        return False, None
    try:
        metadata = read_json(PREPROCESSED_METADATA_PATH)
        expected = expected_preprocessed_static_metadata(len(slice_records_df))
        for key, value in expected.items():
            if metadata.get(key) != value:
                raise RuntimeError(f"Metadata preprocesada incompatible: {key}")
        image_sha256 = sha256_stream(PREPROCESSED_IMAGE_PATH)
        mask_sha256 = sha256_stream(PREPROCESSED_MASK_PATH)
        if metadata.get("image_file_sha256") != image_sha256 or metadata.get("mask_file_sha256") != mask_sha256:
            raise RuntimeError("SHA-256 de memmaps preprocesados incompatible.")
        image_memmap = validate_memmap_file(PREPROCESSED_IMAGE_PATH, tuple(expected["image_shape"]), np.dtype("uint8"))
        mask_memmap = validate_memmap_file(PREPROCESSED_MASK_PATH, tuple(expected["mask_shape"]), np.dtype("uint8"))
        validate_mask_values(mask_memmap)
        del image_memmap
        del mask_memmap
        metadata["preprocessed_cache_content_sha256"] = preprocessed_cache_content_sha256(metadata)
        return True, metadata
    except Exception as exc:
        print(f"Cache preprocesado invalido; se reconstruira: {exc}")
        return False, None

def cleanup_preprocessed_tmp() -> None:
    for path in [
        PREPROCESSED_CACHE_DIR / "spider_images_uint8.tmp.npy",
        PREPROCESSED_CACHE_DIR / "spider_masks_uint8.tmp.npy",
        PREPROCESSED_CACHE_DIR / "spider_preprocessed_cache_metadata.tmp.json",
    ]:
        if path.exists():
            path.unlink()

def build_preprocessed_cache() -> dict[str, Any]:
    PREPROCESSED_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cleanup_preprocessed_tmp()
    total_slices = int(len(slice_records_df))
    expected = expected_preprocessed_static_metadata(total_slices)
    image_tmp = PREPROCESSED_CACHE_DIR / "spider_images_uint8.tmp.npy"
    mask_tmp = PREPROCESSED_CACHE_DIR / "spider_masks_uint8.tmp.npy"
    metadata_tmp = PREPROCESSED_CACHE_DIR / "spider_preprocessed_cache_metadata.tmp.json"
    image_mm = np.lib.format.open_memmap(
        image_tmp,
        mode="w+",
        dtype=np.uint8,
        shape=tuple(expected["image_shape"]),
    )
    mask_mm = np.lib.format.open_memmap(
        mask_tmp,
        mode="w+",
        dtype=np.uint8,
        shape=tuple(expected["mask_shape"]),
    )
    total_pairs = int(slice_records_df["pair_index"].nunique())
    processed_pairs = 0
    for pair_index, group in slice_records_df.groupby("pair_index", sort=True):
        first = group.iloc[0]
        image_arr = canonicalize_spider_array(read_array(Path(first.image_local_path)))
        raw_mask_arr = canonicalize_spider_array(read_array(Path(first.mask_local_path)))
        validate_volume_arrays(image_arr, raw_mask_arr)
        mapped_mask = apply_label_map(raw_mask_arr, SPIDER_LABEL_MAP, case_label=f"spider_pair_{pair_index}")
        for rec in group.itertuples(index=False):
            row_index = int(rec.cache_row_index)
            image_slice = extract_slice(image_arr, int(rec.slice_index), int(rec.slice_axis))
            mask_slice = extract_slice(mapped_mask, int(rec.slice_index), int(rec.slice_axis))
            image_mm[row_index] = resize_image_uint8(robust_normalize(image_slice), CFG.TARGET_SIZE)
            mask_uint8 = resize_mask_uint8(mask_slice, CFG.TARGET_SIZE)
            if not set(int(v) for v in np.unique(mask_uint8)).issubset({0, 1, 2, 3}):
                raise RuntimeError("Mascara preprocesada contiene labels fuera de 0..3.")
            mask_mm[row_index] = mask_uint8
        processed_pairs += 1
        if processed_pairs % 25 == 0 or processed_pairs == total_pairs:
            print(f"Cache 2D preprocesado: {processed_pairs}/{total_pairs}")
    image_mm.flush()
    mask_mm.flush()
    del image_mm
    del mask_mm
    if not image_tmp.exists() or image_tmp.stat().st_size <= 0:
        raise RuntimeError("Memmap temporal de imagen invalido.")
    if not mask_tmp.exists() or mask_tmp.stat().st_size <= 0:
        raise RuntimeError("Memmap temporal de mascara invalido.")
    image_check = validate_memmap_file(image_tmp, tuple(expected["image_shape"]), np.dtype("uint8"))
    mask_check = validate_memmap_file(mask_tmp, tuple(expected["mask_shape"]), np.dtype("uint8"))
    validate_mask_values(mask_check)
    del image_check
    del mask_check
    metadata = {
        **expected,
        "image_file_sha256": sha256_stream(image_tmp),
        "mask_file_sha256": sha256_stream(mask_tmp),
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    atomic_write_json(metadata_tmp, metadata)
    if not metadata_tmp.exists() or metadata_tmp.stat().st_size <= 0:
        raise RuntimeError("Metadata temporal preprocesada invalida.")
    os.replace(image_tmp, PREPROCESSED_IMAGE_PATH)
    os.replace(mask_tmp, PREPROCESSED_MASK_PATH)
    os.replace(metadata_tmp, PREPROCESSED_METADATA_PATH)
    cleanup_preprocessed_tmp()
    reused, validated = validate_existing_preprocessed_cache()
    if not reused or validated is None:
        raise RuntimeError("El cache preprocesado reconstruido no paso validacion.")
    return validated

PREPROCESSED_CACHE_REUSED, PREPROCESSED_CACHE_METADATA = validate_existing_preprocessed_cache()
if not PREPROCESSED_CACHE_REUSED:
    PREPROCESSED_CACHE_METADATA = build_preprocessed_cache()
PREPROCESSED_CACHE_METADATA_VALID = PREPROCESSED_CACHE_METADATA is not None
PREPROCESSED_MEMMAPS_VALID = True
PREPROCESSED_CACHE_CONTENT_SHA256 = PREPROCESSED_CACHE_METADATA["preprocessed_cache_content_sha256"]
PREPROCESSED_IMAGE_FILE_SHA256 = PREPROCESSED_CACHE_METADATA["image_file_sha256"]
PREPROCESSED_MASK_FILE_SHA256 = PREPROCESSED_CACHE_METADATA["mask_file_sha256"]
PREPROCESSED_TOTAL_SLICES = int(PREPROCESSED_CACHE_METADATA["total_slices"])
dataloader_uses_medical_volume_reads = False
medical_volumes_opened_once_per_cache_build = True

@dataclass(frozen=True)
class SliceRecord:
    split: str
    pair_index: int
    patient_id: str
    image_local_path: str
    mask_local_path: str
    slice_index: int
    slice_axis: int
    has_foreground: bool
    cache_row_index: int

class SpiderSegmentationDataset(Dataset):
    def __init__(
        self,
        records: list[SliceRecord],
        image_memmap_path: Path,
        mask_memmap_path: Path,
        augment: bool,
        seed: int,
    ):
        self.records = records
        self.image_memmap_path = Path(image_memmap_path)
        self.mask_memmap_path = Path(mask_memmap_path)
        self.augment = augment
        self.seed = seed
        self._epoch = multiprocessing.Value("i", 0)
        self._images_memmap = None
        self._masks_memmap = None

    def set_epoch(self, epoch: int) -> None:
        with self._epoch.get_lock():
            self._epoch.value = int(epoch)

    def current_epoch(self) -> int:
        with self._epoch.get_lock():
            return int(self._epoch.value)

    def _ensure_memmaps(self) -> None:
        if self._images_memmap is None:
            self._images_memmap = np.load(
                self.image_memmap_path,
                mmap_mode="r",
            )
        if self._masks_memmap is None:
            self._masks_memmap = np.load(
                self.mask_memmap_path,
                mmap_mode="r",
            )

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        self._ensure_memmaps()
        rec = self.records[index]
        row_index = int(rec.cache_row_index)
        image_arr = np.array(self._images_memmap[row_index], copy=True).astype(np.float32) / 255.0
        mask_arr = np.array(self._masks_memmap[row_index], copy=True).astype(np.int64)
        if self.augment:
            effective_seed = (
                self.seed
                + self.current_epoch() * 1_000_003
                + int(index)
            )
            image_arr, mask_arr = augment_pair(image_arr, mask_arr, random.Random(effective_seed))
        image_tensor = torch.from_numpy(image_arr.astype(np.float32)).unsqueeze(0)
        mask_tensor = torch.from_numpy(mask_arr.astype(np.int64))
        return image_tensor, mask_tensor

def records_for_split(split: str) -> list[SliceRecord]:
    frame = slice_records_df.loc[slice_records_df["split"] == split].reset_index(drop=True)
    return [SliceRecord(**row._asdict()) for row in frame.itertuples(index=False)]

def worker_init_fn(worker_id: int) -> None:
    worker_seed = CFG.SEED + worker_id
    random.seed(worker_seed)
    np.random.seed(worker_seed)
    torch.manual_seed(worker_seed)

set_seed(CFG.SEED)
train_generator = torch.Generator()
train_generator.manual_seed(CFG.SEED)
train_dataset = SpiderSegmentationDataset(records_for_split("train"), PREPROCESSED_IMAGE_PATH, PREPROCESSED_MASK_PATH, augment=True, seed=CFG.SEED)
val_dataset = SpiderSegmentationDataset(records_for_split("val"), PREPROCESSED_IMAGE_PATH, PREPROCESSED_MASK_PATH, augment=False, seed=CFG.SEED)
test_dataset = SpiderSegmentationDataset(records_for_split("test"), PREPROCESSED_IMAGE_PATH, PREPROCESSED_MASK_PATH, augment=False, seed=CFG.SEED)
loader_kwargs = {
    "batch_size": CFG.BATCH_SIZE,
    "num_workers": CFG.NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": CFG.NUM_WORKERS > 0,
    "drop_last": False,
    "worker_init_fn": worker_init_fn if CFG.NUM_WORKERS > 0 else None,
}
train_loader = DataLoader(train_dataset, shuffle=True, generator=train_generator, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

def validate_batch(loader: DataLoader, split: str) -> None:
    image, mask = next(iter(loader))
    if image.ndim != 4 or mask.ndim != 3:
        raise RuntimeError(f"Batch {split} con ndim invalido.")
    if image.shape[1] != 1 or tuple(image.shape[-2:]) != CFG.TARGET_SIZE or tuple(mask.shape[-2:]) != CFG.TARGET_SIZE:
        raise RuntimeError(f"Batch {split} con shape invalida.")
    if image.dtype != torch.float32 or mask.dtype != torch.int64:
        raise RuntimeError(f"Batch {split} con dtype invalido.")
    if not torch.isfinite(image).all():
        raise RuntimeError(f"Batch {split} contiene imagen no finita.")
    labels = set(int(v) for v in torch.unique(mask).cpu().tolist())
    if not labels.issubset({0, 1, 2, 3}):
        raise RuntimeError(f"Batch {split} contiene labels fuera de 0..3.")

for split_name, loader in [("train", train_loader), ("val", val_loader), ("test", test_loader)]:
    validate_batch(loader, split_name)

def benchmark_dataloader(dataset: Dataset, max_batches: int = 50) -> dict[str, Any]:
    benchmark_loader = DataLoader(
        dataset,
        batch_size=CFG.BATCH_SIZE,
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=False,
        drop_last=False,
        shuffle=False,
        worker_init_fn=worker_init_fn if CFG.NUM_WORKERS > 0 else None,
    )
    start = time.time()
    processed_batches = 0
    for processed_batches, _batch in enumerate(benchmark_loader, start=1):
        if processed_batches >= max_batches:
            break
    elapsed_seconds = time.time() - start
    seconds_per_batch = elapsed_seconds / max(processed_batches, 1)
    estimated_batches_per_epoch = math.ceil(len(dataset) / CFG.BATCH_SIZE)
    result = {
        "processed_batches": int(processed_batches),
        "elapsed_seconds": float(elapsed_seconds),
        "seconds_per_batch": float(seconds_per_batch),
        "estimated_loading_minutes_per_epoch": float(seconds_per_batch * estimated_batches_per_epoch / 60.0),
        "backend": DATASET_BACKEND,
    }
    if result["processed_batches"] != max_batches:
        raise RuntimeError("Benchmark de DataLoader no proceso 50 batches.")
    if not math.isfinite(result["elapsed_seconds"]) or result["elapsed_seconds"] <= 0:
        raise RuntimeError("Benchmark de DataLoader con elapsed_seconds invalido.")
    if not math.isfinite(result["seconds_per_batch"]) or result["seconds_per_batch"] <= 0:
        raise RuntimeError("Benchmark de DataLoader con seconds_per_batch invalido.")
    return result

DATALOADER_BENCHMARK = benchmark_dataloader(train_dataset, 50)
print(json.dumps({
    "train_slices": len(train_dataset),
    "val_slices": len(val_dataset),
    "test_slices": len(test_dataset),
    "dataloader_benchmark": DATALOADER_BENCHMARK,
}, indent=2))


## GPU y modelo

Exige CUDA antes de entrenar, registra información de GPU, instancia `SagittalUNet2D` y valida un forward inicial.

In [ ]:
if "DEVICE" not in globals() or "gpu_info" not in globals():
    raise RuntimeError("DEVICE y gpu_info deben definirse en el gate confirmado antes de cachear datos.")
print(json.dumps(gpu_info, indent=2))

model = SagittalUNet2D(
    in_channels=1,
    num_classes=CFG.NUM_CLASSES,
    base_channels=CFG.BASE_CHANNELS,
).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
param_mb = total_params * 4 / (1024 * 1024)
class_weights = CLASS_WEIGHTS_TENSOR.to(DEVICE)

with torch.no_grad():
    sample_images, _ = next(iter(train_loader))
    logits = model(sample_images.to(DEVICE))
if list(logits.shape) != [sample_images.shape[0], CFG.NUM_CLASSES, CFG.TARGET_SIZE[0], CFG.TARGET_SIZE[1]]:
    raise RuntimeError("Forward pre-training devolvio shape inesperada.")
if not torch.isfinite(logits).all():
    raise RuntimeError("Forward pre-training produjo logits no finitos.")
print(json.dumps({"total_params": total_params, "trainable_params": trainable_params, "params_mb": param_mb}, indent=2))

## Loss, optimizador y métricas

Combina CrossEntropy ponderada con Dice loss sin fondo, y calcula métricas por clase más macro sin background.

In [ ]:
def dice_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    probs = torch.softmax(logits, dim=1)
    one_hot = F.one_hot(target, num_classes=CFG.NUM_CLASSES).permute(0, 3, 1, 2).float()
    dims = (0, 2, 3)
    intersection = torch.sum(probs * one_hot, dims)
    cardinality = torch.sum(probs + one_hot, dims)
    dice = (2.0 * intersection + 1e-6) / (cardinality + 1e-6)
    return 1.0 - dice[1:].mean()

ce_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

def combined_loss(logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return 0.5 * ce_loss_fn(logits, target) + 0.5 * dice_loss(logits, target)

def metrics_from_confusion(confusion: np.ndarray) -> dict[str, Any]:
    per_class = {}
    dice_values = []
    iou_values = []
    for class_id, class_name in enumerate(CLASS_NAMES):
        tp = float(confusion[class_id, class_id])
        fp = float(confusion[:, class_id].sum() - tp)
        fn = float(confusion[class_id, :].sum() - tp)
        gt_pixels = float(confusion[class_id, :].sum())
        pred_pixels = float(confusion[:, class_id].sum())
        dice_den = 2 * tp + fp + fn
        iou_den = tp + fp + fn
        precision_den = tp + fp
        recall_den = tp + fn
        dice = None if dice_den == 0 else (2 * tp) / dice_den
        iou = None if iou_den == 0 else tp / iou_den
        precision = None if precision_den == 0 else tp / precision_den
        recall = None if recall_den == 0 else tp / recall_den
        per_class[class_name] = {
            "dice": dice,
            "iou": iou,
            "precision": precision,
            "recall": recall,
            "gt_pixels": int(gt_pixels),
            "pred_pixels": int(pred_pixels),
        }
        if class_id > 0 and dice is not None:
            dice_values.append(float(dice))
        if class_id > 0 and iou is not None:
            iou_values.append(float(iou))
    return {
        "per_class": per_class,
        "dice_macro_no_bg": float(np.mean(dice_values)) if dice_values else None,
        "iou_macro_no_bg": float(np.mean(iou_values)) if iou_values else None,
    }

@torch.no_grad()
def evaluate(loader: DataLoader, split: str) -> dict[str, Any]:
    model.eval()
    total_loss = 0.0
    total_batches = 0
    confusion = np.zeros((CFG.NUM_CLASSES, CFG.NUM_CLASSES), dtype=np.int64)
    gt_present_cases = np.zeros(CFG.NUM_CLASSES, dtype=np.int64)
    pred_present_cases = np.zeros(CFG.NUM_CLASSES, dtype=np.int64)
    pred_min = CFG.NUM_CLASSES
    pred_max = 0
    for images, masks in loader:
        images = images.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = combined_loss(logits, masks)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Loss no finita en evaluacion {split}.")
        preds = torch.argmax(logits, dim=1)
        pred_min = min(pred_min, int(preds.min().item()))
        pred_max = max(pred_max, int(preds.max().item()))
        if pred_min < 0 or pred_max >= CFG.NUM_CLASSES:
            raise RuntimeError("Predicciones fuera de 0..3.")
        gt_np = masks.detach().cpu().numpy()
        pred_np = preds.detach().cpu().numpy()
        for sample_index in range(gt_np.shape[0]):
            sample_gt = gt_np[sample_index]
            sample_pred = pred_np[sample_index]
            for class_id in range(CFG.NUM_CLASSES):
                gt_present_cases[class_id] += int(
                    np.any(sample_gt == class_id)
                )
                pred_present_cases[class_id] += int(
                    np.any(sample_pred == class_id)
                )
        flat_gt = gt_np.ravel()
        flat_pred = pred_np.ravel()
        encoded = flat_gt * CFG.NUM_CLASSES + flat_pred
        confusion += np.bincount(encoded, minlength=CFG.NUM_CLASSES * CFG.NUM_CLASSES).reshape(CFG.NUM_CLASSES, CFG.NUM_CLASSES)
        total_loss += float(loss.item())
        total_batches += 1
    if total_batches <= 0:
        raise RuntimeError(f"Evaluacion {split} sin batches.")
    dataset_len = len(loader.dataset)
    if np.any(gt_present_cases < 0) or np.any(gt_present_cases > dataset_len):
        raise RuntimeError("gt_present_cases fuera del rango esperado por muestra.")
    if np.any(pred_present_cases < 0) or np.any(pred_present_cases > dataset_len):
        raise RuntimeError("pred_present_cases fuera del rango esperado por muestra.")
    if pred_min < 0 or pred_max >= CFG.NUM_CLASSES:
        raise RuntimeError("prediction_min_class/prediction_max_class fuera de 0..3.")
    result = metrics_from_confusion(confusion)
    result["loss"] = total_loss / total_batches
    result["batches"] = total_batches
    result["prediction_min_class"] = int(pred_min)
    result["prediction_max_class"] = int(pred_max)
    for class_id, class_name in enumerate(CLASS_NAMES):
        result["per_class"][class_name]["gt_present_cases"] = int(gt_present_cases[class_id])
        result["per_class"][class_name]["pred_present_cases"] = int(pred_present_cases[class_id])
    if result["dice_macro_no_bg"] is None or not math.isfinite(result["dice_macro_no_bg"]):
        raise RuntimeError(f"Dice macro no-bg no finito en {split}.")
    return result

## Loop reanudable

Entrena hasta 80 epochs por defecto, guarda checkpoints atómicos `last` y `best`, restaura RNG al reanudar y aplica early stopping.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=CFG.SCHEDULER_FACTOR,
    patience=CFG.SCHEDULER_PATIENCE,
)
amp_enabled = bool(CFG.USE_AMP and DEVICE.type == "cuda")
try:
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)

if "LAST_CHECKPOINT" not in globals() or "BEST_CHECKPOINT" not in globals():
    raise RuntimeError("Las rutas de checkpoint deben definirse antes del cache completo.")

def rng_state_payload() -> dict[str, Any]:
    return {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state(),
        "torch_cuda_rng_state_all": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
        "train_generator_state": train_generator.get_state(),
    }

def restore_rng_state(payload: dict[str, Any]) -> None:
    random.setstate(payload["python_random_state"])
    np.random.set_state(payload["numpy_random_state"])
    torch.set_rng_state(payload["torch_rng_state"])
    if torch.cuda.is_available() and payload.get("torch_cuda_rng_state_all"):
        torch.cuda.set_rng_state_all(payload["torch_cuda_rng_state_all"])
    train_generator.set_state(payload["train_generator_state"])

def checkpoint_payload(epoch: int, best_val_score: float, bad_epochs: int, history: list[dict[str, Any]], resume_kind: str) -> dict[str, Any]:
    return {
        "checkpoint_version": "gcs_spider_final_v1",
        "epoch": int(epoch),
        "model_key": "sagittal_spider",
        "plane": "sagittal",
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_val_score": float(best_val_score),
        "bad_epochs": int(bad_epochs),
        "history": history,
        "class_weights": CLASS_WEIGHTS_TENSOR.cpu().tolist(),
        "label_group_mapping": {str(k): int(v) for k, v in SPIDER_LABEL_MAP.items()},
        "num_classes": CFG.NUM_CLASSES,
        "base_channels": CFG.BASE_CHANNELS,
        "target_size": list(CFG.TARGET_SIZE),
        "seed": CFG.SEED,
        "current_learning_rate": float(optimizer.param_groups[0]["lr"]),
        "patient_split_sha256": PATIENT_SPLIT_SHA256,
        "cache_content_sha256": CACHE_CONTENT_SHA256,
        "cache_receipt_file_sha256": CACHE_RECEIPT_FILE_SHA256,
        "pair_split_content_sha256": PAIR_SPLIT_CONTENT_SHA256,
        "pair_split_file_sha256": PAIR_SPLIT_FILE_SHA256,
        "slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
        "slice_records_file_sha256": SLICE_RECORDS_FILE_SHA256,
        "mapping_sha256": MAPPING_SHA256,
        "preprocessed_cache_content_sha256": PREPROCESSED_CACHE_CONTENT_SHA256,
        "preprocessed_cache_version": PREPROCESSED_CACHE_VERSION,
        "manifest_sha256": CFG.EXPECTED_MANIFEST_SHA256,
        "training_index_sha256": CFG.EXPECTED_TRAINING_INDEX_SHA256,
        "repository_commit": REPOSITORY_COMMIT,
        "model_architecture_sha256": MODEL_ARCHITECTURE_SHA256,
        "training_config": TRAINING_CONFIG,
        "config_fingerprint": CONFIG_FINGERPRINT,
        "rng_states": rng_state_payload(),
        "resume_kind": resume_kind,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }

def atomic_torch_save(payload: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    torch.save(payload, tmp_path)
    if not tmp_path.exists() or tmp_path.stat().st_size == 0:
        raise RuntimeError("Checkpoint temporal invalido.")
    os.replace(tmp_path, path)

def validate_resume_checkpoint(payload: dict[str, Any], expected_resume_kind: str) -> None:
    required = [
        "epoch", "model_state_dict", "optimizer_state_dict", "scheduler_state_dict", "scaler_state_dict",
        "best_val_score", "bad_epochs", "history", "manifest_sha256", "training_index_sha256",
        "patient_split_sha256", "cache_content_sha256", "cache_receipt_file_sha256",
        "pair_split_content_sha256", "pair_split_file_sha256", "slice_records_content_sha256",
        "slice_records_file_sha256", "mapping_sha256", "preprocessed_cache_content_sha256",
        "preprocessed_cache_version", "repository_commit", "model_architecture_sha256",
        "num_classes", "base_channels", "target_size", "seed", "training_config", "config_fingerprint",
        "rng_states", "checkpoint_version", "model_key", "plane", "resume_kind", "class_weights",
    ]
    if expected_resume_kind == "last":
        required += ["best_epoch", "best_checkpoint_sha256"]
    missing = [key for key in required if key not in payload]
    if missing:
        raise RuntimeError(f"Checkpoint de reanudacion incompleto: {missing}")
    class_weights_payload = payload["class_weights"]
    checks = {
        "checkpoint_version": payload["checkpoint_version"] == "gcs_spider_final_v1",
        "model_key": payload["model_key"] == "sagittal_spider",
        "plane": payload["plane"] == "sagittal",
        "resume_kind_allowed": payload["resume_kind"] in {"last", "best"},
        "resume_kind_expected": payload["resume_kind"] == expected_resume_kind,
        "class_weights_len": len(class_weights_payload) == CFG.NUM_CLASSES,
        "class_weights_finite": all(math.isfinite(float(value)) for value in class_weights_payload),
        "best_val_score_finite": math.isfinite(float(payload["best_val_score"])),
        "epoch_valid": int(payload["epoch"]) >= 1,
        "bad_epochs_valid": int(payload["bad_epochs"]) >= 0,
        "training_config": payload["training_config"] == TRAINING_CONFIG,
        "manifest": payload["manifest_sha256"] == CFG.EXPECTED_MANIFEST_SHA256,
        "index": payload["training_index_sha256"] == CFG.EXPECTED_TRAINING_INDEX_SHA256,
        "patient_split": payload["patient_split_sha256"] == PATIENT_SPLIT_SHA256,
        "cache_content": payload["cache_content_sha256"] == CACHE_CONTENT_SHA256,
        "pair_split_content": payload["pair_split_content_sha256"] == PAIR_SPLIT_CONTENT_SHA256,
        "slice_records_content": payload["slice_records_content_sha256"] == SLICE_RECORDS_CONTENT_SHA256,
        "mapping": payload["mapping_sha256"] == MAPPING_SHA256,
        "preprocessed_cache": payload["preprocessed_cache_content_sha256"] == PREPROCESSED_CACHE_CONTENT_SHA256,
        "preprocessed_cache_version": payload["preprocessed_cache_version"] == PREPROCESSED_CACHE_VERSION,
        "repository": payload["repository_commit"] == REPOSITORY_COMMIT,
        "architecture": payload["model_architecture_sha256"] == MODEL_ARCHITECTURE_SHA256,
        "num_classes": payload["num_classes"] == CFG.NUM_CLASSES,
        "base_channels": payload["base_channels"] == CFG.BASE_CHANNELS,
        "target_size": tuple(payload["target_size"]) == CFG.TARGET_SIZE,
        "seed": payload["seed"] == CFG.SEED,
        "config": payload["config_fingerprint"] == CONFIG_FINGERPRINT,
    }
    failed = [name for name, ok in checks.items() if not ok]
    if failed:
        raise RuntimeError(f"Checkpoint incompatible para reanudar: {failed}")

def validate_best_last_consistency(last_checkpoint: dict[str, Any]) -> dict[str, Any]:
    if not BEST_CHECKPOINT.exists():
        raise RuntimeError("Falta BEST_CHECKPOINT requerido por last checkpoint.")
    observed_best_sha256 = sha256_stream(BEST_CHECKPOINT)
    if observed_best_sha256 != last_checkpoint["best_checkpoint_sha256"]:
        raise RuntimeError("BEST_CHECKPOINT no coincide con best_checkpoint_sha256 del last checkpoint.")
    best_payload = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=False)
    validate_resume_checkpoint(best_payload, "best")
    if int(best_payload["epoch"]) != int(last_checkpoint["best_epoch"]):
        raise RuntimeError("best epoch inconsistente entre last y best checkpoint.")
    if not math.isclose(
        float(best_payload["best_val_score"]),
        float(last_checkpoint["best_val_score"]),
        rel_tol=1e-12,
        abs_tol=1e-12,
    ):
        raise RuntimeError("best_val_score inconsistente entre last y best checkpoint.")
    history_rows = list(last_checkpoint["history"])
    matching_history = [row for row in history_rows if int(row.get("epoch", -1)) == int(last_checkpoint["best_epoch"])]
    if not matching_history:
        raise RuntimeError("best_epoch no aparece en history del last checkpoint.")
    if not math.isclose(
        float(matching_history[-1]["val_dice_macro_no_bg"]),
        float(last_checkpoint["best_val_score"]),
        rel_tol=1e-12,
        abs_tol=1e-12,
    ):
        raise RuntimeError("history no coincide con best_val_score del last checkpoint.")
    return best_payload

history: list[dict[str, Any]] = []
best_val_score = -1.0
bad_epochs = 0
start_epoch = 1
resume_used = False
reason_finished = "max_epochs"
best_last_consistency_verified = False
best_checkpoint_sha256 = ""
current_best_epoch = 0

if CFG.RESUME_TRAINING and LAST_CHECKPOINT.exists():
    checkpoint = torch.load(LAST_CHECKPOINT, map_location=DEVICE, weights_only=False)
    validate_resume_checkpoint(checkpoint, "last")
    validate_best_last_consistency(checkpoint)
    best_checkpoint_sha256 = sha256_stream(BEST_CHECKPOINT)
    best_last_consistency_verified = True
    current_best_epoch = int(checkpoint["best_epoch"])
    model.load_state_dict(checkpoint["model_state_dict"], strict=True)
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
    scaler.load_state_dict(checkpoint["scaler_state_dict"])
    restore_rng_state(checkpoint["rng_states"])
    history = list(checkpoint["history"])
    best_val_score = float(checkpoint["best_val_score"])
    bad_epochs = int(checkpoint["bad_epochs"])
    start_epoch = int(checkpoint["epoch"]) + 1
    resume_used = True
if start_epoch > CFG.MAX_EPOCHS:
    reason_finished = "resumed_already_complete"
else:
    for epoch in range(start_epoch, CFG.MAX_EPOCHS + 1):
        train_dataset.set_epoch(epoch)
        epoch_start = time.time()
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        model.train()
        train_loss_total = 0.0
        train_batches = 0
        total_batches = len(train_loader)
        for batch_index, (images, masks) in enumerate(train_loader, start=1):
            if batch_index == 1 or batch_index % CFG.TRAIN_LOG_EVERY_N_BATCHES == 0 or batch_index == total_batches:
                print(f"epoch {epoch} batch {batch_index}/{total_batches}")
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16, enabled=amp_enabled):
                logits = model(images)
                loss = combined_loss(logits, masks)
            if not torch.isfinite(loss):
                raise RuntimeError("Loss no finita durante entrenamiento.")
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.GRADIENT_CLIP_NORM)
            scaler.step(optimizer)
            getattr(scaler, "update")()
            train_loss_total += float(loss.item())
            train_batches += 1
        train_loss = train_loss_total / max(train_batches, 1)
        val_result = evaluate(val_loader, "val")
        scheduler.step(float(val_result["dice_macro_no_bg"]))
        improved = float(val_result["dice_macro_no_bg"]) > best_val_score
        if improved:
            best_val_score = float(val_result["dice_macro_no_bg"])
            bad_epochs = 0
            current_best_epoch = int(epoch)
        else:
            bad_epochs += 1
        epoch_duration = time.time() - epoch_start
        gpu_max_memory_mb = int(torch.cuda.max_memory_allocated() / (1024 * 1024)) if DEVICE.type == "cuda" else 0
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": float(val_result["loss"]),
            "val_dice_macro_no_bg": float(val_result["dice_macro_no_bg"]),
            "val_iou_macro_no_bg": float(val_result["iou_macro_no_bg"]),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "epoch_duration_seconds": float(epoch_duration),
            "gpu_max_memory_mb": gpu_max_memory_mb,
            "improved": bool(improved),
            "best_val_score": float(best_val_score),
            "bad_epochs": int(bad_epochs),
        }
        history.append(row)
        atomic_write_csv(HISTORY_CSV, pd.DataFrame(history))
        atomic_write_json(HISTORY_JSON, {"history": history})
        if improved:
            best_payload = checkpoint_payload(
                epoch,
                best_val_score,
                bad_epochs,
                history,
                "best",
            )
            atomic_torch_save(
                best_payload,
                BEST_CHECKPOINT,
            )
        best_checkpoint_sha256 = sha256_stream(
            BEST_CHECKPOINT
        )
        last_payload = checkpoint_payload(
            epoch,
            best_val_score,
            bad_epochs,
            history,
            "last",
        )
        last_payload["best_checkpoint_sha256"] = (
            best_checkpoint_sha256
        )
        last_payload["best_epoch"] = (
            current_best_epoch
        )
        atomic_torch_save(
            last_payload,
            LAST_CHECKPOINT,
        )
        best_last_consistency_verified = True
        print(json.dumps({
            "epoch": epoch,
            "train_loss": train_loss,
            "validation_loss": val_result["loss"],
            "validation_dice_macro_no_bg": val_result["dice_macro_no_bg"],
            "learning_rate": optimizer.param_groups[0]["lr"],
            "best_score": best_val_score,
            "bad_epochs": bad_epochs,
            "duration_seconds": epoch_duration,
        }, indent=2))
        if bad_epochs >= CFG.EARLY_STOPPING_PATIENCE:
            reason_finished = "early_stopping"
            break

epochs_completed = len(history)
best_epoch = int(max(history, key=lambda row: row["val_dice_macro_no_bg"])["epoch"]) if history else 0
if epochs_completed == 0 and reason_finished != "resumed_already_complete":
    raise RuntimeError("No se completo ningun epoch.")


## Test final y checkpoint runtime

Carga el mejor checkpoint, evalúa test una sola vez, guarda el `.pt` final compatible con `build_checkpoint_model` y valida strict load en CPU.

In [ ]:
if not LAST_CHECKPOINT.exists() or not BEST_CHECKPOINT.exists():
    raise RuntimeError("Faltan checkpoints last/best.")
best_checkpoint_sha256 = sha256_stream(BEST_CHECKPOINT)
best_checkpoint = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=False)
validate_resume_checkpoint(best_checkpoint, "best")
model.load_state_dict(best_checkpoint["model_state_dict"], strict=True)
test_result = evaluate(test_loader, "test")
if not all(math.isfinite(float(v)) for v in [test_result["loss"], test_result["dice_macro_no_bg"], test_result["iou_macro_no_bg"]]):
    raise RuntimeError("Metricas de test no finitas.")
FINAL_TEST_METRICS_JSON = CFG.OUTPUT_DIR / "final_test_metrics.json"
test_patient_count = int(slice_records_df.loc[slice_records_df["split"] == "test", "patient_id"].nunique())
test_slice_count = int((slice_records_df["split"] == "test").sum())
final_test_metrics = {
    "per_class": test_result["per_class"],
    "dice_macro_no_bg": float(test_result["dice_macro_no_bg"]),
    "iou_macro_no_bg": float(test_result["iou_macro_no_bg"]),
    "test_loss": float(test_result["loss"]),
    "selected_epoch": int(best_checkpoint["epoch"]),
    "best_validation_score": float(best_checkpoint["best_val_score"]),
    "test_patients": test_patient_count,
    "test_slices": test_slice_count,
    "manifest_sha256": CFG.EXPECTED_MANIFEST_SHA256,
    "training_index_sha256": CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "patient_split_sha256": PATIENT_SPLIT_SHA256,
    "slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
    "slice_records_file_sha256": SLICE_RECORDS_FILE_SHA256,
}
atomic_write_json(FINAL_TEST_METRICS_JSON, final_test_metrics)

FINAL_MODEL_PATH = CFG.OUTPUT_DIR / "sagittal_spider_final_v1.pt"
final_payload = {
    "model_state_dict": best_checkpoint["model_state_dict"],
    "num_classes": CFG.NUM_CLASSES,
    "base_channels": CFG.BASE_CHANNELS,
    "target_size": list(CFG.TARGET_SIZE),
    "class_weights": CLASS_WEIGHTS_TENSOR.cpu().tolist(),
    "classes": CLASS_NAMES,
    "label_group_mapping": {str(k): int(v) for k, v in SPIDER_LABEL_MAP.items()},
    "sagittal_axis": SPIDER_SAGITTAL_AXIS,
    "slice_strategy": {
        "max_foreground_slices_per_volume": CFG.MAX_FOREGROUND_SLICES_PER_VOLUME,
        "background_ratio": CFG.BACKGROUND_RATIO,
        "max_background_if_no_foreground": CFG.MAX_BACKGROUND_IF_NO_FOREGROUND,
    },
    "metrics": final_test_metrics,
    "model_key": "sagittal_spider",
    "plane": "sagittal",
    "seed": CFG.SEED,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_manifest_sha256": CFG.EXPECTED_MANIFEST_SHA256,
    "source_training_index_sha256": CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "source_patient_split_sha256": PATIENT_SPLIT_SHA256,
    "source_slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
    "source_slice_records_file_sha256": SLICE_RECORDS_FILE_SHA256,
    "source_cache_content_sha256": CACHE_CONTENT_SHA256,
    "source_cache_receipt_file_sha256": CACHE_RECEIPT_FILE_SHA256,
    "source_pair_split_content_sha256": PAIR_SPLIT_CONTENT_SHA256,
    "source_pair_split_file_sha256": PAIR_SPLIT_FILE_SHA256,
    "source_repository_commit": REPOSITORY_COMMIT,
    "source_model_architecture_sha256": MODEL_ARCHITECTURE_SHA256,
    "training_config": TRAINING_CONFIG,
    "resume_metadata": {"resume_used": resume_used, "start_epoch": start_epoch, "reason_finished": reason_finished},
    "warning": "Modelo académico de apoyo. No utilizar como sustituto del criterio profesional ni como dispositivo médico validado.",
}
atomic_torch_save(final_payload, FINAL_MODEL_PATH)

loaded = torch.load(
    FINAL_MODEL_PATH,
    map_location="cpu",
    weights_only=False,
)
strict_model, runtime_meta = build_checkpoint_model(
    "sagittal_spider",
    loaded,
)
strict_model.eval()
with torch.no_grad():
    strict_logits = strict_model(torch.randn(1, 1, 256, 256))
strict_load_output_shape = list(strict_logits.shape)
strict_load_success = strict_load_output_shape == [1, 4, 256, 256] and torch.isfinite(strict_logits).all().item()
if not strict_load_success:
    raise RuntimeError("Strict load del modelo final fallo.")

## Visualizaciones, evidencia y JSON final

Guarda curvas, preview, summary y evidence. `FINAL_TRAINING_SUCCESS` mide éxito técnico; `QUALITY_GATE_PASSED` mide desempeño contra el umbral académico.

In [ ]:
TRAINING_CURVES_PNG = CFG.OUTPUT_DIR / "training_curves.png"
TEST_PREVIEW_PNG = CFG.OUTPUT_DIR / "test_predictions_preview.png"

history_df = pd.DataFrame(history)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(history_df["epoch"], history_df["train_loss"], label="train")
axes[0, 0].plot(history_df["epoch"], history_df["val_loss"], label="val")
axes[0, 0].set_title("Loss")
axes[0, 0].legend()
axes[0, 1].plot(history_df["epoch"], history_df["val_dice_macro_no_bg"])
axes[0, 1].set_title("Validation Dice macro no-bg")
axes[1, 0].plot(history_df["epoch"], history_df["learning_rate"])
axes[1, 0].set_title("Learning rate")
axes[1, 1].axis("off")
fig.tight_layout()
fig.savefig(TRAINING_CURVES_PNG, dpi=140)
plt.close(fig)

def deterministic_preview_records(records: list[SliceRecord], count: int = 6) -> list[SliceRecord]:
    by_patient: dict[str, SliceRecord] = {}
    for rec in records:
        if rec.patient_id not in by_patient:
            by_patient[rec.patient_id] = rec
    values = list(by_patient.values())
    if len(values) < count:
        values = records
    idxs = choose_indices(list(range(len(values))), min(count, len(values)))
    return [values[idx] for idx in idxs]

preview_records = deterministic_preview_records(records_for_split("test"), 6)
fig, axes = plt.subplots(len(preview_records), 3, figsize=(9, 3 * len(preview_records)))
if len(preview_records) == 1:
    axes = np.expand_dims(axes, 0)
model.eval()
for row_idx, rec in enumerate(preview_records):
    image_arr = resize_image(robust_normalize(extract_slice(read_array(Path(rec.image_local_path)), rec.slice_index, rec.slice_axis)), CFG.TARGET_SIZE)
    mask_arr = resize_mask(apply_label_map(extract_slice(read_array(Path(rec.mask_local_path)), rec.slice_index, rec.slice_axis), SPIDER_LABEL_MAP), CFG.TARGET_SIZE)
    tensor = torch.from_numpy(image_arr.astype(np.float32)).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = torch.argmax(model(tensor), dim=1).squeeze(0).cpu().numpy()
    axes[row_idx, 0].imshow(image_arr, cmap="gray")
    axes[row_idx, 0].set_title("imagen")
    axes[row_idx, 1].imshow(mask_arr, vmin=0, vmax=3)
    axes[row_idx, 1].set_title("GT")
    axes[row_idx, 2].imshow(pred, vmin=0, vmax=3)
    axes[row_idx, 2].set_title("pred")
    for col in range(3):
        axes[row_idx, col].axis("off")
fig.tight_layout()
fig.savefig(TEST_PREVIEW_PNG, dpi=140)
plt.close(fig)

QUALITY_GATE_PASSED = bool(float(test_result["dice_macro_no_bg"]) >= QUALITY_GATE_TARGET_DICE_MACRO_NO_BG)

def artifact_hashes(paths: list[Path]) -> dict[str, str]:
    return {path.name: sha256_stream(path) for path in paths if path.exists()}

SUMMARY_JSON = CFG.OUTPUT_DIR / "gcs_spider_final_training_summary.json"
EVIDENCE_JSON = CFG.OUTPUT_DIR / "gcs_spider_final_training_evidence.json"
summary = {
    "manifest_sha256": CFG.EXPECTED_MANIFEST_SHA256,
    "training_index_sha256": CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "repository_commit": REPOSITORY_COMMIT,
    "architecture_sha256": MODEL_ARCHITECTURE_SHA256,
    "index_rows": int(len(spider_index_df)),
    "unique_patients": int(len(patient_ids)),
    "patient_counts": {k: int(v) for k, v in patient_counts.items()},
    "legacy_v4_slice_counts": {k: int(v) for k, v in LEGACY_V4_SLICE_COUNTS.items()},
    "actual_slice_counts": {k: int(v) for k, v in slice_counts.items()},
    "slice_counts_match_legacy_v4": bool(slice_counts_match_legacy_v4),
    "slice_strategy_changed_from_legacy": True,
    "slice_strategy_change_reason": "Las slices background excluyen todo indice con foreground.",
    "cached_pairs": int(len(cache_receipt_df)),
    "downloaded_or_reused_objects": int(verified_cache_objects),
    "verified_cache_objects": int(verified_cache_objects),
    "device": str(DEVICE),
    "gpu_name": gpu_info.get("gpu_name", ""),
    "amp_enabled": amp_enabled,
    "base_channels": CFG.BASE_CHANNELS,
    "batch_size": CFG.BATCH_SIZE,
    "target_size": list(CFG.TARGET_SIZE),
    "num_classes": CFG.NUM_CLASSES,
    "start_epoch": int(start_epoch),
    "epochs_completed": int(epochs_completed),
    "best_epoch": int(best_checkpoint["epoch"]),
    "best_validation_dice_macro_no_bg": float(best_checkpoint["best_val_score"]),
    "test_loss": float(test_result["loss"]),
    "test_dice_macro_no_bg": float(test_result["dice_macro_no_bg"]),
    "test_iou_macro_no_bg": float(test_result["iou_macro_no_bg"]),
    "quality_gate_target": QUALITY_GATE_TARGET_DICE_MACRO_NO_BG,
    "QUALITY_GATE_PASSED": QUALITY_GATE_PASSED,
    "final_model_saved": FINAL_MODEL_PATH.exists(),
    "strict_load_success": bool(strict_load_success),
    "strict_load_output_shape": strict_load_output_shape,
    "checkpoint_last_saved": LAST_CHECKPOINT.exists(),
    "checkpoint_best_saved": BEST_CHECKPOINT.exists(),
    "training_curves_saved": TRAINING_CURVES_PNG.exists(),
    "test_preview_saved": TEST_PREVIEW_PNG.exists(),
    "resume_used": resume_used,
    "reason_finished": reason_finished,
    "gcs_write_operations": GCS_WRITE_OPERATIONS,
    "cache_content_sha256": CACHE_CONTENT_SHA256,
    "cache_receipt_file_sha256": CACHE_RECEIPT_FILE_SHA256,
    "pair_split_content_sha256": PAIR_SPLIT_CONTENT_SHA256,
    "pair_split_file_sha256": PAIR_SPLIT_FILE_SHA256,
    "slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
    "slice_records_file_sha256": SLICE_RECORDS_FILE_SHA256,
    "best_checkpoint_sha256": best_checkpoint_sha256,
    "best_last_consistency_verified": bool(best_last_consistency_verified),
    "stable_resume_fingerprints": True,
    "preprocessed_cache_version": PREPROCESSED_CACHE_VERSION,
    "preprocessed_cache_content_sha256": PREPROCESSED_CACHE_CONTENT_SHA256,
    "preprocessed_image_file_sha256": PREPROCESSED_IMAGE_FILE_SHA256,
    "preprocessed_mask_file_sha256": PREPROCESSED_MASK_FILE_SHA256,
    "preprocessed_total_slices": PREPROCESSED_TOTAL_SLICES,
    "dataloader_benchmark": DATALOADER_BENCHMARK,
    "dataloader_uses_medical_volume_reads": dataloader_uses_medical_volume_reads,
    "medical_volumes_opened_once_per_cache_build": medical_volumes_opened_once_per_cache_build,
    "orphan_checkpoint_guard": bool(orphan_checkpoint_guard),
    "portable_cache_path_fingerprints": bool(portable_cache_path_fingerprints),
}
summary["FINAL_TRAINING_SUCCESS"] = bool(
    checks and all(checks.values())
    and int(len(spider_index_df)) == CFG.EXPECTED_INDEX_ROWS
    and int(len(patient_ids)) == CFG.EXPECTED_UNIQUE_PATIENTS
    and patient_counts == EXPECTED_PATIENT_COUNTS
    and all_splits_have_slices
    and bool(all_splits_have_foreground)
    and slice_records_no_leakage
    and background_slices_verified_empty
    and bool(patient_split_matches_reconstruction)
    and bool(pair_split_matches_reconstruction)
    and bool(slice_records_match_reconstruction)
    and len(CACHE_CONTENT_SHA256) == 64
    and len(PAIR_SPLIT_CONTENT_SHA256) == 64
    and len(SLICE_RECORDS_CONTENT_SHA256) == 64
    and bool(orphan_checkpoint_guard)
    and bool(portable_cache_path_fingerprints)
    and bool(best_last_consistency_verified)
    and bool(PREPROCESSED_CACHE_METADATA_VALID)
    and bool(PREPROCESSED_MEMMAPS_VALID)
    and PREPROCESSED_TOTAL_SLICES == len(slice_records_df)
    and len(PREPROCESSED_CACHE_CONTENT_SHA256) == 64
    and dataloader_uses_medical_volume_reads is False
    and verified_cache_objects == CFG.EXPECTED_SPIDER_OBJECTS
    and set(SPIDER_LABEL_MAP.values()) == {0, 1, 2, 3}
    and torch.isfinite(CLASS_WEIGHTS_TENSOR).all().item()
    and DEVICE.type == "cuda"
    and (epochs_completed > 0 or reason_finished == "resumed_already_complete")
    and LAST_CHECKPOINT.exists()
    and BEST_CHECKPOINT.exists()
    and FINAL_TEST_METRICS_JSON.exists()
    and all(math.isfinite(float(v)) for v in [test_result["loss"], test_result["dice_macro_no_bg"], test_result["iou_macro_no_bg"]])
    and FINAL_MODEL_PATH.exists()
    and strict_load_success
    and HISTORY_CSV.exists()
    and HISTORY_JSON.exists()
    and TRAINING_CURVES_PNG.exists()
    and TEST_PREVIEW_PNG.exists()
    and GCS_WRITE_OPERATIONS == 0
)
if not summary["FINAL_TRAINING_SUCCESS"]:
    raise RuntimeError("FINAL_TRAINING_SUCCESS no cumple todas las condiciones tecnicas.")
atomic_write_json(SUMMARY_JSON, summary)

final_artifacts = [
    CACHE_RECEIPT_CSV,
    MAPPING_JSON,
    PATIENT_SPLIT_CSV,
    PAIR_SPLIT_CSV,
    SLICE_RECORDS_CSV,
    CLASS_DISTRIBUTION_JSON,
    HISTORY_CSV,
    HISTORY_JSON,
    LAST_CHECKPOINT,
    BEST_CHECKPOINT,
    FINAL_MODEL_PATH,
    FINAL_TEST_METRICS_JSON,
    TRAINING_CURVES_PNG,
    TEST_PREVIEW_PNG,
    SUMMARY_JSON,
]
evidence = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "45_gcs_spider_final_training.ipynb",
    "repository_commit": REPOSITORY_COMMIT,
    "architecture_sha256": MODEL_ARCHITECTURE_SHA256,
    "preflight_commit": "351da8cec1aa41de1d5426b5d530b8c53a5af7fa",
    "smoke_commit": "903817bad5099dcb5172d619076e4e8b175bc57e",
    "manifest_sha256": CFG.EXPECTED_MANIFEST_SHA256,
    "training_index_sha256": CFG.EXPECTED_TRAINING_INDEX_SHA256,
    "patient_split_sha256": PATIENT_SPLIT_SHA256,
    "cache_content_sha256": CACHE_CONTENT_SHA256,
    "cache_receipt_file_sha256": CACHE_RECEIPT_FILE_SHA256,
    "pair_split_content_sha256": PAIR_SPLIT_CONTENT_SHA256,
    "pair_split_file_sha256": PAIR_SPLIT_FILE_SHA256,
    "slice_records_content_sha256": SLICE_RECORDS_CONTENT_SHA256,
    "slice_records_file_sha256": SLICE_RECORDS_FILE_SHA256,
    "mapping_sha256": MAPPING_SHA256,
    "config_fingerprint": CONFIG_FINGERPRINT,
    "gpu_information": gpu_info,
    "amp_enabled": amp_enabled,
    "resume_used": resume_used,
    "start_epoch": start_epoch,
    "selected_best_epoch": int(best_checkpoint["epoch"]),
    "reason_finished": reason_finished,
    "verified_gcs_downloads": verified_cache_objects,
    "gcs_read_only": GCS_WRITE_OPERATIONS == 0,
    "gcs_write_operations": GCS_WRITE_OPERATIONS,
    "best_checkpoint_sha256": best_checkpoint_sha256,
    "best_last_consistency_verified": bool(best_last_consistency_verified),
    "stable_resume_fingerprints": True,
    "preprocessed_cache_version": PREPROCESSED_CACHE_VERSION,
    "preprocessed_cache_content_sha256": PREPROCESSED_CACHE_CONTENT_SHA256,
    "preprocessed_image_file_sha256": PREPROCESSED_IMAGE_FILE_SHA256,
    "preprocessed_mask_file_sha256": PREPROCESSED_MASK_FILE_SHA256,
    "preprocessed_total_slices": PREPROCESSED_TOTAL_SLICES,
    "dataloader_benchmark": DATALOADER_BENCHMARK,
    "dataloader_uses_medical_volume_reads": dataloader_uses_medical_volume_reads,
    "medical_volumes_opened_once_per_cache_build": medical_volumes_opened_once_per_cache_build,
    "orphan_checkpoint_guard": bool(orphan_checkpoint_guard),
    "portable_cache_path_fingerprints": bool(portable_cache_path_fingerprints),
    "legacy_v4_slice_counts": {k: int(v) for k, v in LEGACY_V4_SLICE_COUNTS.items()},
    "actual_slice_counts": {k: int(v) for k, v in slice_counts.items()},
    "slice_counts_match_legacy_v4": bool(slice_counts_match_legacy_v4),
    "slice_strategy_changed_from_legacy": True,
    "slice_strategy_change_reason": "Las slices background excluyen todo indice con foreground.",
    "background_slices_verified_empty": bool(background_slices_verified_empty),
    "presence_metrics_counted_per_sample": True,
    "gpu_checked_before_cache": True,
    "confirmation_token_stored": False,
    "artifact_sha256": artifact_hashes(final_artifacts),
}
atomic_write_json(EVIDENCE_JSON, evidence)
print(json.dumps(summary, indent=2))